# Step 0: Importing Api and Packages

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
sns.set('notebook', font_scale=1.25, style='whitegrid')

from pathlib import Path
import re

import nba_api
import nba_api.stats.endpoints as ep
from nba_api.stats.static import teams

import xgboost as xgb

import re
import unicodedata

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
from nba_api.stats.endpoints import (
    leaguedashplayerstats,
    PlayerEstimatedMetrics,
    ScheduleLeagueV2,
    LeagueDashPlayerBioStats,
    GravityLeaders,
    LeagueDashPlayerShotLocations,
    LeagueDashPlayerStats,
    LeagueDashPtDefend,
    LeagueDashPlayerPtShot,
    ScheduleLeagueV2,
)

# Seasons to pull
SEASONS = [
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25",
    "2025-26",
    "2026-27",
]

endpoints = {
    "ScheduleLeagueV2": ScheduleLeagueV2,
    "LeagueDashPlayerBioStats": LeagueDashPlayerBioStats,
    "GravityLeaders": GravityLeaders,
    "LeagueDashPlayerShotLocations": LeagueDashPlayerShotLocations,
    "LeagueDashPlayerStats": LeagueDashPlayerStats,
    "LeagueDashPtDefend": LeagueDashPtDefend,
    "LeagueDashPlayerPtShot": LeagueDashPlayerPtShot,
    "ScheduleLeagueV2": ScheduleLeagueV2,
}

## Retrieving Data 

# Step 1: Retrieve Dataframes From NBA API

To improve performance, I’ll pull the remote API dataframes locally so they can be accessed more efficiently during manipulation and analysis.

In [3]:
# OUTPUT_DIR = "/Users/ronanhwang/Downloads/Github/nba_2020s/csv_files"

# os.makedirs(OUTPUT_DIR, exist_ok = True)

# # Helper function to convert SEASONS to seasons in api
# def season_filename(season, endpoint_name):
#     start, end = season.split("-")
#     season_short = f"{start[-2:]}_{end[-2:]}"
#     return f"{season_short}{endpoint_name}.csv"


# for season in SEASONS:

#     print(f"SEASON: {season}")

#     for name, endpoint in endpoints.items():

#         filename = season_filename(season, name)

#         filepath = os.path.join(OUTPUT_DIR, filename)

#         print(f"\nPulling {name}...")

#         try:
#             # Different endpoints have different args
#             if name == "ScheduleLeagueV2":

#                 response = endpoint(
#                     league_id="00",
#                     season=season,
#                 )

#             else:

#                 response = endpoint(
#                     season=season,
#                     season_type_all_star="Regular Season",
#                 )

#             # Get DataFrame
#             df = response.get_data_frames()[0]

#             # Export
#             df.to_csv(filepath, index = False)

#             print(f"Exported: {filename}")
#             print(f"Shape: {df.shape}")

#             # Read CSV back in
#             df_check = pd.read_csv(filepath)

#             print(f"\nHead of {filename}:")
#             print(df_check.head())

#             # Look for season columns
#             season_cols = [
#                 col
#                 for col in df_check.columns
#                 if "SEASON" in col.upper()
#                 or "YEAR" in col.upper()
#             ]

#             if season_cols:
#                 print("\nSeason information:")

#                 for col in season_cols:
#                     print(
#                         f"  {col}: "
#                         f"{df_check[col].dropna().unique()[:10]}"
#                     )

#         except Exception as e:

#             print(f"ERROR pulling {name} for {season}:")
#             print(e)


# Step 2: Read in Data 

In [4]:
CSV_DIR = Path("/Users/ronanhwang/Downloads/Github/nba_2020s/csv_files")


# Helper function that laods all NBA CSV files from csv_files directory
# Expects 23_24LeagueDashPlayerStats.csv format
# Returns 
    # data = {
        # "23_24": {
        #     "LeagueDashPlayerStats": DataFrame,
        #     "GravityLeaders": DataFrame,
        #     ...
        # },
def load_nba_csvs(csv_dir):
    data = {}

    # Loop through every CSV in the directory
    for filepath in csv_dir.glob("*.csv"):

        filename = filepath.name

        # Extract season from beginning of filename
        match = re.match(r"^(\d{2}_\d{2})(.+)\.csv$", filename)

        if not match:
            print(f"Skipping unrecognized filename: {filename}")
            continue

        season = match.group(1)
        endpoint = match.group(2)

        # Create season dictionary if necessary
        if season not in data:
            data[season] = {}

        # Read CSV
        df = pd.read_csv(filepath)

        # Store DataFrame
        data[season][endpoint] = df

        print(f"{filename} -> {season} / {endpoint} | Shape: {df.shape}")

    return data


In [5]:
data = load_nba_csvs(CSV_DIR)

26_27LeagueDashPlayerBioStats.csv -> 26_27 / LeagueDashPlayerBioStats | Shape: (582, 23)
24_25ScheduleLeagueV2.csv -> 24_25 / ScheduleLeagueV2 | Shape: (1400, 159)
22_23LeagueDashPtDefend.csv -> 22_23 / LeagueDashPtDefend | Shape: (535, 14)
21_22LeagueDashPlayerStats.csv -> 21_22 / LeagueDashPlayerStats | Shape: (605, 67)
26_27LeagueDashPlayerShotLocations.csv -> 26_27 / LeagueDashPlayerShotLocations | Shape: (583, 30)
23_24LeagueDashPlayerShotLocations.csv -> 23_24 / LeagueDashPlayerShotLocations | Shape: (573, 30)
23_24LeagueDashPlayerPtShot.csv -> 23_24 / LeagueDashPlayerPtShot | Shape: (568, 20)
26_27NBA2k.csv -> 26_27 / NBA2k | Shape: (650, 46)
23_24ScheduleLeagueV2.csv -> 23_24 / ScheduleLeagueV2 | Shape: (1396, 184)
21_22LeagueDashPlayerPtShot.csv -> 21_22 / LeagueDashPlayerPtShot | Shape: (596, 20)
25_26Misc.csv -> 25_26 / Misc | Shape: (582, 20)
23_24LeagueDashPlayerBioStats.csv -> 23_24 / LeagueDashPlayerBioStats | Shape: (572, 23)
25_26LeagueDashPlayerPtShot.csv -> 25_26 / L

## See all .csv files 

In [6]:
for season, endpoints in data.items():
    print(f"\n{season}")

    for endpoint, df in endpoints.items():
        print(f"  {endpoint}: {df.shape}")


26_27
  LeagueDashPlayerBioStats: (582, 23)
  LeagueDashPlayerShotLocations: (583, 30)
  NBA2k: (650, 46)
  LeagueDashPtDefend: (580, 14)
  ScheduleLeagueV2: (1274, 160)
  Rosters: (596, 8)
  GravityLeaders: (234, 27)
  LeagueDashPlayerStats: (582, 69)
  LeagueDashPlayerPtShot: (582, 20)
  Misc: (582, 20)

24_25
  ScheduleLeagueV2: (1400, 159)
  LeagueDashPlayerBioStats: (569, 23)
  LeagueDashPlayerShotLocations: (570, 30)
  LeagueDashPlayerPtShot: (566, 20)
  LeagueDashPlayerStats: (569, 67)
  LeagueDashPtDefend: (568, 14)
  Misc: (569, 20)

22_23
  LeagueDashPtDefend: (535, 14)
  ScheduleLeagueV2: (1394, 151)
  LeagueDashPlayerStats: (539, 67)
  Misc: (539, 20)
  LeagueDashPlayerPtShot: (537, 20)
  LeagueDashPlayerBioStats: (539, 23)
  LeagueDashPlayerShotLocations: (540, 30)

21_22
  LeagueDashPlayerStats: (605, 67)
  LeagueDashPlayerPtShot: (596, 20)
  LeagueDashPtDefend: (595, 14)
  LeagueDashPlayerBioStats: (605, 23)
  ScheduleLeagueV2: (1393, 152)
  LeagueDashPlayerShotLocation

# Step 2.5: Update 2026_27 Data

In [7]:
# Helper function that updates the teams using rosters taken from nba.com
def update_teams_from_rosters(data, season="26_27"):
    rosters = data[season]["Rosters"].copy()
    stats = data[season]["LeagueDashPlayerStats"].copy()

    # Team ids
    team_id_map = {
        "ATL": 1610612737,
        "BOS": 1610612738,
        "CLE": 1610612739,
        "NOP": 1610612740,
        "CHI": 1610612741,
        "DAL": 1610612742,
        "DEN": 1610612743,
        "GSW": 1610612744,
        "HOU": 1610612745,
        "LAC": 1610612746,
        "LAL": 1610612747,
        "MIA": 1610612748,
        "MIL": 1610612749,
        "MIN": 1610612750,
        "BKN": 1610612751,
        "NYK": 1610612752,
        "ORL": 1610612753,
        "IND": 1610612754,
        "PHI": 1610612755,
        "PHX": 1610612756,
        "POR": 1610612757,
        "SAC": 1610612758,
        "SAS": 1610612759,
        "OKC": 1610612760,
        "TOR": 1610612761,
        "UTA": 1610612762,
        "MEM": 1610612763,
        "WAS": 1610612764,
        "DET": 1610612765,
        "CHA": 1610612766,
    }

    # Clean player names
    rosters["Player_clean"] = (
        rosters["Player"]
        .astype(str)
        .str.replace("\n", " ", regex=False)
        .str.strip()
    )

    stats["PLAYER_NAME_clean"] = (
        stats["PLAYER_NAME"]
        .astype(str)
        .str.strip()
    )

    # Build player and team
    roster_lookup = (
        rosters
        .drop_duplicates("Player_clean")
        .set_index("Player_clean")
    )

    # Match stats to player
    mapped_team = stats["PLAYER_NAME_clean"].map(
        roster_lookup["Team"]
    )

    matched = mapped_team.notna()
    unmatched = ~matched

    # Update team abbv and id and match player to team, FA if none
    stats["TEAM_ABBREVIATION"] = mapped_team

    stats.loc[unmatched, "TEAM_ABBREVIATION"] = "FA"

    stats["TEAM_ID"] = (
        stats["TEAM_ABBREVIATION"]
        .map(team_id_map)
        .fillna(0)
        .astype("int64")
    )

    # Add roster players missing from stats
    stats_names = set(stats["PLAYER_NAME_clean"])
    roster_names = set(rosters["Player_clean"])

    missing_in_stats = roster_names - stats_names

    if missing_in_stats:
        # Previous season
        previous_season = f"{int(season[:2]) - 1:02d}_{int(season[3:]) - 1:02d}"

        # Example:
        # 26_27 -> 25_26
        if season == "26_27":
            previous_season = "24_25"

        previous_stats = data[previous_season]["LeagueDashPlayerStats"].copy()

        previous_stats["PLAYER_NAME_clean"] = (
            previous_stats["PLAYER_NAME"]
            .astype(str)
            .str.strip()
        )

        previous_lookup = (
            previous_stats
            .drop_duplicates("PLAYER_NAME_clean")
            .set_index("PLAYER_NAME_clean")
        )

        # Numeric columns in current stats
        numeric_cols = stats.select_dtypes(
            include="number"
        ).columns.tolist()

        new_rows = []

        for name in missing_in_stats:

            # Current roster team
            team = roster_lookup.loc[name, "Team"]
            team_id = team_id_map.get(team, 0)

            # Case 1: Player existed in previous season
            if name in previous_lookup.index:

                previous_row = previous_lookup.loc[name].copy()

                if isinstance(previous_row, pd.DataFrame):
                    previous_row = previous_row.iloc[0]

                # Start with an empty current-season row
                row = pd.Series(index=stats.columns, dtype="object")

                # Copy over whatever previous-season stats exist
                for col in stats.columns:
                    if col in previous_row.index:
                        row[col] = previous_row[col]

                # Current season player and team info
                row["PLAYER_NAME"] = name
                row["TEAM_ABBREVIATION"] = team
                row["TEAM_ID"] = team_id

                if "TEAM" in row.index:
                    row["TEAM"] = team

                if "Team" in row.index:
                    row["Team"] = team

            # Case 2: Player did NOT exist in previous season either
            else:

                row = pd.Series(index=stats.columns, dtype="object")

                row["PLAYER_NAME"] = name
                row["TEAM_ABBREVIATION"] = team
                row["TEAM_ID"] = team_id
                row["AGE"] = 0

                # Zero out numeric stats
                for col in numeric_cols:
                    row[col] = 0

            new_rows.append(row)

        # Append all new rows at once
        if new_rows:
            new_rows_df = pd.DataFrame(new_rows)

            stats = pd.concat(
                [stats, new_rows_df],
                ignore_index=True
            )

    # Clean temp
    stats.drop(
        columns="PLAYER_NAME_clean",
        inplace=True,
        errors="ignore"
    )

    # Save
    data[season]["LeagueDashPlayerStats"] = stats

    return data

In [8]:
data = update_teams_from_rosters(data, "26_27")

# Step 3: Cleaning Data

In [9]:
# Helper function that cleans all NBA DataFrames stored in the data dictionary by dropping/renaming
# Expects: data[season][endpoint] = DataFrame
# Returns: Cleaned data dictionary.
def clean_nba_data(data):
    
    # Columns to drop from LeagueDashPlayerStats
    player_stats_drop = [
        'WNBA_FANTASY_PTS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK',
        'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK',
        'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK',
        'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK',
        'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK',
        'PTS_RANK', 'PLUS_MINUS_RANK', 'NBA_FANTASY_PTS_RANK', 'DD2_RANK',
        'TD3_RANK', 'WNBA_FANTASY_PTS_RANK', 'FP_HIGH_SCORE_RANK',
        'TEAM_COUNT'
    ]

    # Columns to drop from LeagueDashPtDefend
    pt_defend_drop = [
        'AGE', 'GP', 'G'
    ]

    # Rename LeagueDashPtDefend columns
    pt_defend_rename = {
        "CLOSE_DEF_PERSON_ID": "D_PLAYER_ID",
        "PLAYER_NAME": "D_PLAYER_NAME",
        "PLAYER_LAST_TEAM_ID": "D_PLAYER_LAST_TEAM_ID",
        "PLAYER_LAST_TEAM_ABBREVIATION": "D_PLAYER_LAST_TEAM_ABBREVIATION",
        "PLAYER_POSITION": "D_PLAYER_POSITION",
        "FREQ": "D_FREQ",
        "NORMAL_FG_PCT": "D_NORMAL_FG_PCT",
        "PCT_PLUSMINUS": "D_PCT_PLUSMINUS",
    }

    # Rename LeagueDashPlayerShotLocations columns
    shot_locations_rename = {
        "Unnamed: 0": "PLAYER_ID",
        "Unnamed: 1": "PLAYER_NAME",
        "Unnamed: 2": "TEAM_ID",
        "Unnamed: 3": "TEAM_ABBREVIATION",
        "Unnamed: 4": "AGE",
        "Unnamed: 5": "NICKNAME",
    }

    # Columns to drop from LeagueDashPlayerPtShot
    pt_shot_drop = [
        'AGE', 'GP', 'FGM', 'FGA',
        'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT'
    ]

    # Columns to keep from ScheduleLeagueV2
    schedule_keep = [
        'seasonYear',
        'gameDate',
        'gameId',
        'gameCode',
        'gameStatusText',
        'gameSequence',
        'gameDateTimeEst',
        'seriesGameNumber',
        'gameLabel',
        'gameSubLabel',
        'seriesText',
        'postponedStatus',
        'isNeutral',
        'homeTeam_teamId',
        'homeTeam_teamTricode',
        'homeTeam_wins',
        'homeTeam_losses',
        'homeTeam_score',
        'awayTeam_teamId',
        'awayTeam_teamTricode',
        'awayTeam_wins',
        'awayTeam_losses',
        'awayTeam_score'
    ]

    # Columns to keep from Misc
    misc_keep = [
        'Player',
        'TEAM',
        'AGE',
        'PTS\nOFF TO',
        '2nd\nPTS',
        'FBPs',
        'PITP',
        'Opp PTS\nOFF TO',
        'Opp\n2nd PTS',
        'Opp\nFBPs',
        'Opp\nPITP',
        'BLK',
        'BLKA'
    ]


    # Loop through every season
    for season, endpoints in data.items():

        print(f"\nCleaning {season}...")

        # LeagueDashPlayerStats
        if "LeagueDashPlayerStats" in endpoints:
            endpoints["LeagueDashPlayerStats"].drop(
                columns=player_stats_drop,
                errors="ignore",
                inplace=True
            )
        
        # LeagueDashPtDefend
        if "LeagueDashPtDefend" in endpoints:
            endpoints["LeagueDashPtDefend"].drop(
                columns=pt_defend_drop,
                errors="ignore",
                inplace=True
            )

            endpoints["LeagueDashPtDefend"].rename(
                columns=pt_defend_rename,
                inplace=True
            )
        
        # LeagueDashPlayerShotLocations
        if "LeagueDashPlayerShotLocations" in endpoints:
            endpoints["LeagueDashPlayerShotLocations"].rename(
                columns=shot_locations_rename,
                inplace=True
            )

            # Shot-location NaNs represent no recorded shots in that category
            endpoints["LeagueDashPlayerShotLocations"].fillna(0, inplace=True)
        
        # LeagueDashPlayerPtShot
        if "LeagueDashPlayerPtShot" in endpoints:
            endpoints["LeagueDashPlayerPtShot"].drop(
                columns=pt_shot_drop,
                errors="ignore",
                inplace=True
            )

        # ScheduleLeagueV2
        if "ScheduleLeagueV2" in endpoints:
            df = endpoints["ScheduleLeagueV2"]

            endpoints["ScheduleLeagueV2"] = df[
                [
                    col
                    for col in schedule_keep
                    if col in df.columns
                ]
            ]

        # Misc
        if "Misc" in endpoints:
            df = endpoints["Misc"]

            # Keep only desired columns
            endpoints["Misc"] = df[
                [
                    col
                    for col in misc_keep
                    if col in df.columns
                ]
            ]

            # Fill missing stat values with 0
            misc_stat_columns = [
                col for col in misc_keep
                if col not in ["Player", "TEAM"]
                and col in endpoints["Misc"].columns
            ]

            endpoints["Misc"][misc_stat_columns] = (
                endpoints["Misc"][misc_stat_columns].fillna(0)
            )

        print(f"Finished {season}")

    return data


Note that the data is cleaned within the notebook and is not updated in the original files. Therefore, the folders in the directory still contain the original, uncleaned data.

In [10]:
data = clean_nba_data(data)


Cleaning 26_27...
Finished 26_27

Cleaning 24_25...
Finished 24_25

Cleaning 22_23...
Finished 22_23

Cleaning 21_22...
Finished 21_22

Cleaning 23_24...
Finished 23_24

Cleaning 25_26...
Finished 25_26


/var/folders/0b/n3f548tn3156qpndhm9hlkhh0000gn/T/ipykernel_9238/2089117680.py:172: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  endpoints["Misc"][misc_stat_columns] = (
/var/folders/0b/n3f548tn3156qpndhm9hlkhh0000gn/T/ipykernel_9238/2089117680.py:172: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  endpoints["Misc"][misc_stat_columns] = (
/var/folders/0b/n3f548tn3156qpndhm9hlkhh0000gn/T/ipykernel_9238/2089117680.py:172: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFram

# Step 4: Merging Dataframes Together

In [11]:
# Helper function that merges Stats, Shot Locations, Defense,
# Player Shot data, and Misc data for every season.
# Expected structure: data[season][endpoint] = DataFrame
# Returns: merged_data[season] = merged DataFrame
def merge_nba_data(data):

    merged_data = {}

    shot_columns = [
        "Restricted Area", "Restricted Area.1", "Restricted Area.2",
        "In The Paint (Non-RA)", "In The Paint (Non-RA).1", "In The Paint (Non-RA).2",
        "Mid-Range", "Mid-Range.1", "Mid-Range.2",
        "Left Corner 3", "Left Corner 3.1", "Left Corner 3.2",
        "Right Corner 3", "Right Corner 3.1", "Right Corner 3.2",
        "Above the Break 3", "Above the Break 3.1", "Above the Break 3.2",
        "Backcourt", "Backcourt.1", "Backcourt.2",
        "Corner 3", "Corner 3.1", "Corner 3.2"
    ]

    # Normalize player names ONLY for matching across datasets
    def normalize_player_name(name):

        if pd.isna(name):
            return name

        name = str(name).strip()

        # Normalize Jr. -> Jr
        name = name.replace(" Jr.", " Jr")

        # Known cross-source naming difference
        name_overrides = {
            "Bobby Portis Jr": "Bobby Portis",
            "Brandon Boston Jr": "Brandon Boston",
            "Jonas Valanciunas": "Jonas Valančiūnas",
            "Lester Quiñones": "Lester Quinones",
            "Vlatko Cancar": "Vlatko Čančar",
        }

        return name_overrides.get(name, name)

    for season, endpoints in data.items():

        print(f"\nMerging {season}...")

        # Get current-season DataFrames
        stats = endpoints.get("LeagueDashPlayerStats")
        shots = endpoints.get("LeagueDashPlayerShotLocations")
        defense = endpoints.get("LeagueDashPtDefend")
        freq = endpoints.get("LeagueDashPlayerPtShot")
        misc = endpoints.get("Misc")

        # Check required DataFrames
        if stats is None:
            print(f"  Skipping {season}: missing Player Stats")
            continue

        if shots is None:
            print(f"  Skipping {season}: missing Shot Locations")
            continue

        if defense is None:
            print(f"  Skipping {season}: missing Defense")
            continue

        if freq is None:
            print(f"  Skipping {season}: missing Player Shot data")
            continue

        if misc is None:
            print(f"  Skipping {season}: missing Misc data")
            continue

        # Players whose data should ALWAYS come from 24_25
        name_overrides2 = {
            "Damian Lillard",
            "Tyrese Haliburton",
            "Kyrie Irving",
            "Jimmy Butler",
            "Ja Morant",
            "Trae Young",
            "Anthony Davis",
            "Fred VanVleet",
            "Domantas Sabonis",
            "Jaren Jackson Jr.",
            "Zach LaVine",
            "Zach Edey",
            "Walker Kessler"
        }

        # Make copies
        stats = stats.copy()
        shots = shots.copy()
        defense = defense.copy()
        freq = freq.copy()
        misc = misc.copy()

        # Use 24_25 data for these override players
        if season != "24_25":

            source_24 = data["24_25"]

            # Get 24_25 DataFrames
            stats_24 = source_24["LeagueDashPlayerStats"].copy()
            shots_24 = source_24["LeagueDashPlayerShotLocations"].copy()
            defense_24 = source_24["LeagueDashPtDefend"].copy()
            freq_24 = source_24["LeagueDashPlayerPtShot"].copy()
            misc_24 = source_24["Misc"].copy()

            # Make sure 24_25 IDs are numeric too
            stats_24["PLAYER_ID"] = pd.to_numeric(
                stats_24["PLAYER_ID"], errors="coerce"
            ).astype("Int64")

            shots_24["PLAYER_ID"] = pd.to_numeric(
                shots_24["PLAYER_ID"], errors="coerce"
            ).astype("Int64")

            defense_24["D_PLAYER_ID"] = pd.to_numeric(
                defense_24["D_PLAYER_ID"], errors="coerce"
            ).astype("Int64")

            freq_24["PLAYER_ID"] = pd.to_numeric(
                freq_24["PLAYER_ID"], errors="coerce"
            ).astype("Int64")

            # Find override players in 24_25 Stats
            override_stats = stats_24[
                stats_24["PLAYER_NAME"].isin(name_overrides2)
            ].copy()

            override_ids = set(
                override_stats["PLAYER_ID"].dropna()
            )

            if len(override_stats) > 0:
                
                # Stats
                # Remove current-season versions
                stats = stats[
                    ~stats["PLAYER_NAME"].isin(name_overrides2)
                ].copy()

                # Add 24_25 versions
                stats = pd.concat(
                    [stats, override_stats],
                    ignore_index=True
                )

                # Shot locations
                override_shots = shots_24[
                    shots_24["PLAYER_ID"].isin(override_ids)
                ].copy()

                # Remove current-season versions
                shots = shots[
                    ~shots["PLAYER_ID"].isin(override_ids)
                ].copy()

                # Add 24_25 versions
                shots = pd.concat(
                    [shots, override_shots],
                    ignore_index=True
                )

                # Defense
                override_defense = defense_24[
                    defense_24["D_PLAYER_ID"].isin(override_ids)
                ].copy()

                defense = defense[
                    ~defense["D_PLAYER_ID"].isin(override_ids)
                ].copy()

                defense = pd.concat(
                    [defense, override_defense],
                    ignore_index=True
                )

                # Player Shot/ Frequency
                override_freq = freq_24[
                    freq_24["PLAYER_ID"].isin(override_ids)
                ].copy()

                freq = freq[
                    ~freq["PLAYER_ID"].isin(override_ids)
                ].copy()

                freq = pd.concat(
                    [freq, override_freq],
                    ignore_index=True
                )

                # Misc
                misc_24["_MISC_PLAYER_NAME"] = (
                    misc_24["Player"]
                    .apply(normalize_player_name)
                )

                override_names = {
                    normalize_player_name(name)
                    for name in name_overrides2
                }

                override_misc = misc_24[
                    misc_24["_MISC_PLAYER_NAME"].isin(override_names)
                ].copy()

                misc["_MISC_PLAYER_NAME"] = (
                    misc["Player"]
                    .apply(normalize_player_name)
                )

                misc = misc[
                    ~misc["_MISC_PLAYER_NAME"].isin(override_names)
                ].copy()

                misc = pd.concat(
                    [misc, override_misc],
                    ignore_index=True
                )
        # Make player id constant

        stats["PLAYER_ID"] = pd.to_numeric(
            stats["PLAYER_ID"],
            errors="coerce"
        ).astype("Int64")

        shots["PLAYER_ID"] = pd.to_numeric(
            shots["PLAYER_ID"],
            errors="coerce"
        ).astype("Int64")

        defense["D_PLAYER_ID"] = pd.to_numeric(
            defense["D_PLAYER_ID"],
            errors="coerce"
        ).astype("Int64")

        freq["PLAYER_ID"] = pd.to_numeric(
            freq["PLAYER_ID"],
            errors="coerce"
        ).astype("Int64")

        # Clean player names
        stats["PLAYER_NAME"] = (
            stats["PLAYER_NAME"]
            .astype(str)
            .str.strip()
        )

        misc["Player"] = (
            misc["Player"]
            .astype(str)
            .str.strip()
        )

        # Stats + Shot Locations
        merged = stats.merge(
            shots,
            on="PLAYER_ID",
            how="left",
            suffixes=("", "_SHOT")
        )

        existing_shot_columns = [
            col for col in shot_columns
            if col in merged.columns
        ]

        merged[existing_shot_columns] = (
            merged[existing_shot_columns].fillna(0)
        )

        # Merge Defense
        merged = merged.merge(
            defense,
            left_on="PLAYER_ID",
            right_on="D_PLAYER_ID",
            how="left"
        )

        # Merge Player Shot/Frequency Data
        merged = merged.merge(
            freq,
            on="PLAYER_ID",
            how="left",
            suffixes=("", "_CLUTCH")
        )

        # Normalize names for Misc merge
        merged["_MISC_PLAYER_NAME"] = (
            merged["PLAYER_NAME"]
            .apply(normalize_player_name)
        )

        misc["_MISC_PLAYER_NAME"] = (
            misc["Player"]
            .apply(normalize_player_name)
        )

        # Merge Misc
        merged = merged.merge(
            misc,
            on="_MISC_PLAYER_NAME",
            how="left",
            suffixes=("", "_MISC")
        )

        merged.drop(
            columns="_MISC_PLAYER_NAME",
            inplace=True
        )

        # Store merged DataFrame
        merged_data[season] = merged

        print(f"  Merged shape: {merged.shape}")

    return merged_data

In [12]:
merged_data = merge_nba_data(data)


Merging 26_27...
  Merged shape: (697, 100)

Merging 24_25...
  Merged shape: (569, 99)

Merging 22_23...
  Merged shape: (552, 99)

Merging 21_22...
  Merged shape: (617, 99)

Merging 23_24...
  Merged shape: (584, 99)

Merging 25_26...
  Merged shape: (594, 100)


# Step 5: Check Missing Data

In [13]:
# Helper function that inspects rows containing NaN values across all merged seasons
# Expected structure: merged_data[season] = DataFrame
# Returns: None
def inspect_missing_rows(merged_data):

    for season, df in merged_data.items():

        # Find columns containing NaN
        missing_counts = df.isna().sum()
        missing_columns = missing_counts[missing_counts > 0]

        if missing_columns.empty:
            print("\nNo missing values.")
            continue

        # Print rows containing missing values
        rows_with_nan = df[df.isna().any(axis=1)]

        print(f"Rows with missing values: {len(rows_with_nan):,}")

        print(rows_with_nan.to_string(index=False))

In [14]:
inspect_missing_rows(merged_data)

Rows with missing values: 128
 PLAYER_ID        PLAYER_NAME NICKNAME    TEAM_ID TEAM_ABBREVIATION  AGE  GP  W  L  W_PCT         MIN  FGM  FGA  FG_PCT  FG3M  FG3A  FG3_PCT  FTM  FTA  FT_PCT  OREB  DREB  REB  AST  TOV  STL  BLK  BLKA  PF  PFD  PTS  PLUS_MINUS  NBA_FANTASY_PTS  DD2  TD3  FP_HIGH_SCORE  PLAYER_NAME_SHOT TEAM_ID_SHOT TEAM_ABBREVIATION_SHOT AGE_SHOT NICKNAME_SHOT Restricted Area Restricted Area.1 Restricted Area.2 In The Paint (Non-RA) In The Paint (Non-RA).1 In The Paint (Non-RA).2 Mid-Range Mid-Range.1 Mid-Range.2 Left Corner 3 Left Corner 3.1 Left Corner 3.2 Right Corner 3 Right Corner 3.1 Right Corner 3.2 Above the Break 3 Above the Break 3.1 Above the Break 3.2 Backcourt Backcourt.1 Backcourt.2 Corner 3 Corner 3.1 Corner 3.2  D_PLAYER_ID     D_PLAYER_NAME  D_PLAYER_LAST_TEAM_ID D_PLAYER_LAST_TEAM_ABBREVIATION D_PLAYER_POSITION  D_FREQ  D_FGM  D_FGA  D_FG_PCT  D_NORMAL_FG_PCT  D_PCT_PLUSMINUS PLAYER_NAME_CLUTCH  PLAYER_LAST_TEAM_ID PLAYER_LAST_TEAM_ABBREVIATION    G  FGA

# Step 6: Create Dataframe with Winning features

In [15]:
col_Altogether = [

    # Player stats
    'PLAYER_ID',
    'PLAYER_NAME',
    'TEAM_ID',
    'TEAM_ABBREVIATION',

    'AGE',
    'GP',
    'W',
    'L',
    'W_PCT',
    'MIN',

    'FGM',
    'FGA',
    'FG_PCT',

    'FG3M',
    'FG3A',
    'FG3_PCT',

    'FTM',
    'FTA',
    'FT_PCT',

    'OREB',
    'DREB',
    'REB',
    'AST',
    'TOV',
    'STL',
    'BLK',
    'BLKA',
    'PF',
    'PFD',

    'PTS',
    'PLUS_MINUS',
    'NBA_FANTASY_PTS',
    'DD2',
    'TD3',

    # Shot locations
    # Base   = FGM
    # .1     = FGA
    # .2     = FG%
    'Restricted Area',
    'Restricted Area.1',
    'Restricted Area.2',

    'In The Paint (Non-RA)',
    'In The Paint (Non-RA).1',
    'In The Paint (Non-RA).2',

    'Mid-Range',
    'Mid-Range.1',
    'Mid-Range.2',

    'Left Corner 3',
    'Left Corner 3.1',
    'Left Corner 3.2',

    'Right Corner 3',
    'Right Corner 3.1',
    'Right Corner 3.2',

    'Above the Break 3',
    'Above the Break 3.1',
    'Above the Break 3.2',

    'Backcourt',
    'Backcourt.1',
    'Backcourt.2',

    # Aggregate of Left Corner 3 + Right Corner 3
    'Corner 3',
    'Corner 3.1',
    'Corner 3.2',

    # Defense
    'D_PLAYER_POSITION',
    'D_FREQ',
    'D_FGM',
    'D_FGA',
    'D_FG_PCT',
    'D_NORMAL_FG_PCT',
    'D_PCT_PLUSMINUS',

    # Player shot
    'FGA_FREQUENCY',
    'EFG_PCT',
    'FG2A_FREQUENCY',
    'FG2M',
    'FG2A',
    'FG2_PCT',
    'FG3A_FREQUENCY',

    # Misc
    'PTSOFFTO',
    '2ndPTS',
    'FBPs',
    'PITP',

    'Opp PTSOFF TO',
    'Opp2nd PTS',
    'OppFBPs',
    'OppPITP'
]


In [16]:
# Helper function that creates the final player-level dataset for every season.
# Expected structure: merged_data[season] = DataFrame
# Returns: player_data[season] = transformed DataFrame
def create_player_dataset(merged_data):

    player_data = {}

    for season, df in merged_data.items():

        print(f"\nCreating player dataset for {season}...")

        # Rename miscellaneous source columns
        df = df.rename(columns={
            'PTS\nOFF TO': 'PTSOFFTO',
            '2nd\nPTS': '2ndPTS',
            'Opp PTS\nOFF TO': 'Opp PTSOFF TO',
            'Opp\n2nd PTS': 'Opp2nd PTS',
            'Opp\nFBPs': 'OppFBPs',
            'Opp\nPITP': 'OppPITP'
        })

        # Start with the columns from col_Altogether
        output = df[
            [
                col
                for col in col_Altogether
                if col in df.columns
            ]
        ].copy()

        # Rename columns 
        output = output.rename(columns={
            'PLAYER_ID': 'PlayerID',
            'PLAYER_NAME': 'PlayerName',
            'TEAM_ID': 'TeamID',
            'TEAM_ABBREVIATION': 'TeamAbbv',

            'AGE': 'Age',
            'W_PCT': 'W_Pct',
            'MIN': 'Min',

            'FG3M': '3PM',
            'OREB': 'OReb',
            'DREB': 'DReb',
            'AST': 'Ast',
            'TOV': 'TO',
            'PTS': 'Pts',

            'D_PLAYER_POSITION': 'Pos'
        })

        # Remaining games
        output['RemainingGames'] = output['GP']

        # Paint points
        paint_fgm = (
            output['Restricted Area']
            .fillna(0)
            .astype(float)
            +
            output['In The Paint (Non-RA)']
            .fillna(0)
            .astype(float)
        )

        output['Pts_Paint'] = paint_fgm * 2

        # Mid-range points
        midrange_fgm = (
            output['Mid-Range']
            .fillna(0)
            .astype(float)
        )

        output['Pts_MidRange'] = midrange_fgm * 2

        # 3-point points
        three_fgm = (
            output['Left Corner 3']
            .fillna(0)
            .astype(float)
            +
            output['Right Corner 3']
            .fillna(0)
            .astype(float)
            +
            output['Above the Break 3']
            .fillna(0)
            .astype(float)
            +
            output['Backcourt']
            .fillna(0)
            .astype(float)
        )

        output['Pts_3PT'] = three_fgm * 3

        # Store transformed player-level dataset
        player_data[season] = output
    return player_data


In [17]:
player_data = create_player_dataset(merged_data)


Creating player dataset for 26_27...

Creating player dataset for 24_25...

Creating player dataset for 22_23...

Creating player dataset for 21_22...

Creating player dataset for 23_24...

Creating player dataset for 25_26...


# Step 7: Create Dataframe with Player Per Game Averages

In [18]:
# Features in the dataframe that I want to shift to PerGame and PerSeason
per_game_cols = [
    'Min', 'FGM', 'FGA', '3PM', 'FG3A',
    'FTM', 'FTA',
    'OReb', 'DReb', 'REB',
    'Ast', 'TO', 'STL', 'BLK', 'BLKA',
    'PF', 'PFD', 'Pts',
    'NBA_FANTASY_PTS', 'DD2', 'TD3',
    'Pts_Paint', 'Pts_MidRange', 'Pts_3PT', 'PLUS_MINUS'
]

feature_cols = [
    'PlayerID', 'PlayerName', 'TeamID', 'TeamAbbv', 'Age', 'GP',
    'W', 'L', 'W_Pct',
    'FG_PCT', 'FG3_PCT', 'FT_PCT',
    'Restricted Area', 'Restricted Area.1', 'Restricted Area.2',
    'In The Paint (Non-RA)', 'In The Paint (Non-RA).1', 'In The Paint (Non-RA).2',
    'Mid-Range', 'Mid-Range.1', 'Mid-Range.2',
    'Left Corner 3', 'Left Corner 3.1', 'Left Corner 3.2',
    'Right Corner 3', 'Right Corner 3.1', 'Right Corner 3.2',
    'Above the Break 3', 'Above the Break 3.1', 'Above the Break 3.2',
    'Backcourt', 'Backcourt.1', 'Backcourt.2',
    'Corner 3', 'Corner 3.1', 'Corner 3.2',
    'Pos',
    'D_FREQ', 'D_FGM', 'D_FGA', 'D_FG_PCT', 'D_NORMAL_FG_PCT',
    'D_PCT_PLUSMINUS',
    'FGA_FREQUENCY', 'EFG_PCT',
    'FG2A_FREQUENCY', 'FG2M', 'FG2A', 'FG2_PCT',
    'FG3A_FREQUENCY',
    'PTSOFFTO', '2ndPTS', 'FBPs', 'PITP',
    'Opp PTSOFF TO', 'Opp2nd PTS', 'OppFBPs', 'OppPITP',
    'RemainingGames'
]

In [19]:
# Helper function that creates a per-game version of each season's player dataset.
#     Counting stats are divided by GP.
#     All other feature columns are carried over unchanged.
# Expected structure: data[season] = DataFrame
# Returns: per_game_data[season] = DataFrame
def create_per_game_avg(data):
    per_game_data = {}

    for season, df in data.items():

        print(f"Creating per-game averages for {season}...")

        # Only keep columns that actually exist
        available_features = [
            col for col in feature_cols
            if col in df.columns
        ]

        available_per_game = [
            col for col in per_game_cols
            if col in df.columns
        ]

        # Start with non-per-game features
        output = df[available_features].copy()

        # Calculate per-game averages
        for col in available_per_game:

            output[col] = (
                pd.to_numeric(df[col], errors='coerce')
                / pd.to_numeric(df['GP'], errors='coerce')
            )

        per_game_data[season] = output

    return per_game_data


In [20]:
PerGameAvg = create_per_game_avg(player_data)

Creating per-game averages for 26_27...
Creating per-game averages for 24_25...
Creating per-game averages for 22_23...
Creating per-game averages for 21_22...
Creating per-game averages for 23_24...
Creating per-game averages for 25_26...


# Step 8: Fill in Rookies

In [21]:
# Copies so we don't accidentally modify the originals
pga_nba2k = PerGameAvg["26_27"].copy()
nba2k = data["26_27"]["NBA2k"].copy()


In [22]:
# Helper function that normalizes player names
def clean_player_name(name):
    if pd.isna(name):
        return ""

    name = str(name)

    # Remove accents
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))

    # Lowercase
    name = name.lower()

    # Replace common punctuation with spaces
    name = re.sub(r"[.'’`]", "", name)
    name = re.sub(r"[-]", " ", name)

    # Remove suffixes
    name = re.sub(r"\b(jr|sr|ii|iii|iv|v)\b", "", name)

    # Remove anything that isn't a letter/number/space
    name = re.sub(r"[^a-z0-9 ]", "", name)

    # Collapse whitespace
    name = re.sub(r"\s+", " ", name).strip()

    return name


In [23]:
# Use nba 2k stats to fill rookies data
pga_nba2k["PlayerName_clean"] = pga_nba2k["PlayerName"].apply(clean_player_name)
nba2k["name_clean"] = nba2k["name"].apply(clean_player_name)

pga_nba2k = pga_nba2k.merge(
    nba2k,
    left_on="PlayerName_clean",
    right_on="name_clean",
    how="left"
)

pga_nba2k.drop(
    columns=["PlayerName_clean", "name_clean"],
    inplace=True
)

# Remove players without an NBA 2K match
pga_nba2k = pga_nba2k[
    pga_nba2k["overall"].notna()
].copy()

pga_nba2k = pga_nba2k.drop(columns=[
    'PlayerID', 'PlayerName', 'TeamID', 'TeamAbbv',
    'Age', 'W', 'L', 'W_Pct', 'name', 'slug', 'team', 'era'
])

In [24]:
# Helper function that converts feet to inches. Used for dataframe merging and conversions
def height_to_inches(h):
    if pd.isna(h):
        return np.nan

    h = str(h).strip()

    # Handles formats like 6'11" or 6'11
    match = re.match(r"^(\d+)'(\d+)", h)

    if match:
        feet = int(match.group(1))
        inches = int(match.group(2))
        return feet * 12 + inches

    return np.nan

pga_nba2k['height'] = pga_nba2k['height'].apply(height_to_inches)

In [25]:
# Helper function that assigns primary and secondary positions to numerical values
position_map = {
    'PG': 1,
    'SG': 2,
    'SF': 3,
    'PF': 4,
    'C': 5
}

def split_positions(pos):
    if pd.isna(pos):
        return np.nan, np.nan
    
    pos = str(pos).upper().strip()
    
    # Handle separators such as "-", "/", ",", etc.
    parts = re.split(r'[-/, ]+', pos)
    parts = [p for p in parts if p in position_map]
    
    if len(parts) == 0:
        return np.nan, np.nan
    
    primary = position_map[parts[0]]
    secondary = position_map[parts[1]] if len(parts) > 1 else np.nan
    
    return primary, secondary


pga_nba2k[['primarypos', 'secondarypos']] = (
    pga_nba2k['positions']
    .apply(lambda x: pd.Series(split_positions(x)))
)

In [26]:
# Input features and target features
input_features = [
    'primarypos',
    'secondarypos',
    'height',
    'overall',
    'close_shot',
    'mid_range_shot',
    'three_ball',
    'free_throw',
    'shot_iq',
    'offensive_consistency',
    'driving_layup',
    'standing_dunk',
    'dunk',
    'post_hook',
    'post_fade',
    'post_control',
    'draw_foul',
    'hands',
    'pass_accuracy',
    'ball_handle',
    'speed_with_ball',
    'pass_iq',
    'pass_vision',
    'speed',
    'agility',
    'vertical',
    'strength',
    'stamina',
    'hustle',
    'durability',
    'interior_defense',
    'defense',
    'steal',
    'block',
    'help_defense_iq',
    'pass_perception',
    'defensive_consistency',
    'offensive_rebound',
    'defensive_rebound'
]

target_features = [
    'GP',
    'FG_PCT',
    'FG3_PCT',
    'FT_PCT',
    'Restricted Area',
    'Restricted Area.1',
    'Restricted Area.2',
    'In The Paint (Non-RA)',
    'In The Paint (Non-RA).1',
    'In The Paint (Non-RA).2',
    'Mid-Range',
    'Mid-Range.1',
    'Mid-Range.2',
    'Left Corner 3',
    'Left Corner 3.1',
    'Left Corner 3.2',
    'Right Corner 3',
    'Right Corner 3.1',
    'Right Corner 3.2',
    'Above the Break 3',
    'Above the Break 3.1',
    'Above the Break 3.2',
    'Backcourt',
    'Backcourt.1',
    'Backcourt.2',
    'Corner 3',
    'Corner 3.1',
    'Corner 3.2',
    'D_FREQ',
    'D_FGM',
    'D_FGA',
    'D_FG_PCT',
    'D_NORMAL_FG_PCT',
    'D_PCT_PLUSMINUS',
    'FGA_FREQUENCY',
    'EFG_PCT',
    'FG2A_FREQUENCY',
    'FG2M',
    'FG2A',
    'FG2_PCT',
    'FG3A_FREQUENCY',
    'PTSOFFTO',
    '2ndPTS',
    'FBPs',
    'PITP',
    'Opp PTSOFF TO',
    'Opp2nd PTS',
    'OppFBPs',
    'OppPITP',
    'RemainingGames',
    'Min',
    'FGM',
    'FGA',
    '3PM',
    'FG3A',
    'FTM',
    'FTA',
    'OReb',
    'DReb',
    'REB',
    'Ast',
    'TO',
    'STL',
    'BLK',
    'BLKA',
    'PF',
    'PFD',
    'Pts',
    'NBA_FANTASY_PTS',
    'DD2',
    'TD3',
    'Pts_Paint',
    'Pts_MidRange',
    'Pts_3PT',
    'PLUS_MINUS'
]


In [27]:
# Creating model
model_df = pga_nba2k[
    input_features + target_features
].copy()

# Drop players with missing data features
before = len(model_df)

model_df = model_df.dropna(
    subset=input_features
).copy()

after = len(model_df)

# X = All input features
X_all = model_df[input_features].copy()

# Create one fixed train test split
# Every target uses the EXACT SAME players in train/test.
# This lets us evaluate the whole system consistently.
train_idx, test_idx = train_test_split(
    model_df.index,
    test_size=0.20,
    random_state=42
)

X_train_all = X_all.loc[train_idx]
X_test_all  = X_all.loc[test_idx]

# Train one model for eachtarget features
models = {}
predictions = []
results = []


for target in target_features:

    # Target values
    y = model_df[target]

    # Only use players who actually have this target
    train_mask = y.loc[train_idx].notna()
    test_mask = y.loc[test_idx].notna()

    X_train = X_train_all.loc[train_mask]
    y_train = y.loc[train_idx][train_mask]

    X_test = X_test_all.loc[test_mask]
    y_test = y.loc[test_idx][test_mask]

    # Need enough training data
    if len(X_train) < 20:
        print(f"Skipping {target}: only {len(X_train)} training samples")
        continue

    # XGBoost
    model = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1
    )

    # ALL input features go into the model
    model.fit(
        X_train,
        y_train
    )

    # Predict
    y_pred = model.predict(X_test)

    # Metrics
    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            y_pred
        )
    )

    r2 = r2_score(
        y_test,
        y_pred
    )

    # Store model
    models[target] = model

    # Store predictions
    pred_df = pd.DataFrame({
        "PlayerIndex": y_test.index,
        "Target": target,
        "Actual": y_test.values,
        "Predicted": y_pred
    })

    predictions.append(pred_df)

    # Store metrics
    results.append({
        "Target": target,
        "Samples": len(y_test),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

# Combine Predictions
predictions_df = pd.concat(
    predictions,
    ignore_index=True
)

# Results
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "R2",
    ascending=False
).reset_index(drop=True)

print("Model Performance")

print(
    results_df.to_string(index=False)
)

Model Performance
                 Target  Samples       MAE       RMSE        R2
                 D_FREQ       89  0.000000   0.000000  1.000000
          FGA_FREQUENCY       90  0.000000   0.000000  1.000000
                   FG2M       90 30.358836  43.719722  0.877060
                   FG2A       90 58.112618  78.550693  0.873526
                    Pts       92  2.037247   2.740912  0.835631
                    FGM       92  0.710657   0.965202  0.834757
                    Ast       92  0.567180   0.771289  0.829629
        NBA_FANTASY_PTS       92  3.493985   4.729911  0.820516
                    FGA       92  1.393901   2.016612  0.816289
                   PITP       90  1.069111   1.508114  0.796021
    Above the Break 3.1      106 43.455551  62.424546  0.790264
      Above the Break 3      106 15.512074  23.706078  0.787250
                    FTA       92  0.639595   0.881320  0.785848
                  D_FGA       89 96.420051 125.748555  0.779095
                    FT

In [28]:
# Pull Rookies
rookies = PerGameAvg["26_27"][
    PerGameAvg["26_27"]["PlayerID"] == 0
].copy()

# Pull rookie 2k data
rookies_nba2k = nba2k[
    nba2k["name"].isin(rookies["PlayerName"])
].copy()

# Save players name before dropping it
rookie_names = rookies_nba2k["name"].copy()

# Convert Height
rookies_nba2k["height"] = rookies_nba2k["height"].apply(
    height_to_inches
)

# Create numerical primary and secondary positions
rookies_nba2k[["primarypos", "secondarypos"]] = (
    rookies_nba2k["positions"]
    .apply(
        lambda x: pd.Series(
            split_positions(x)
        )
    )
)

# Convert 2k positions to pergame format
def convert_2k_position(pos):

    if pd.isna(pos):
        return np.nan

    pos = float(pos)

    if pos in [1.0, 2.0]:
        return "G"

    elif pos in [3.0, 4.0]:
        return "F"

    elif pos == 5.0:
        return "C"

    return np.nan


def combine_2k_positions(primary, secondary):

    primary_pos = convert_2k_position(primary)
    secondary_pos = convert_2k_position(secondary)

    # No positions
    if pd.isna(primary_pos) and pd.isna(secondary_pos):
        return np.nan

    # Only secondary exists
    if pd.isna(primary_pos):
        return secondary_pos

    # Only primary exists
    if pd.isna(secondary_pos):
        return primary_pos

    # If both map to the same position, only keep one
    if primary_pos == secondary_pos:
        return primary_pos

    # Both exist and are different
    return f"{primary_pos}-{secondary_pos}"

# Save positions before dropping name
rookie_primarypos = rookies_nba2k["primarypos"].copy()
rookie_secondarypos = rookies_nba2k["secondarypos"].copy()

# Drop unused columns 
columns_to_drop = [
    'team',
    'passing',
    'post_moves',
    'slug',
    'name',
    'era',
    'lateral_quickness',
    'positions',
    'name_clean',
    'acceleration'
]

rookies_nba2k = rookies_nba2k.drop(
    columns=columns_to_drop
)

# Check model inputs
missing_inputs = [
    col for col in input_features
    if col not in rookies_nba2k.columns
]

In [29]:
# Model Rookie input
rookie_X = rookies_nba2k[input_features].copy()

# Create transformed df
rookies_transformed = pd.DataFrame(
    index=rookies_nba2k.index
)

# Reattatch player name
rookies_transformed["PlayerName"] = rookie_names

# Reattatch original positions
rookies_transformed["primarypos"] = rookie_primarypos

rookies_transformed["secondarypos"] = rookie_secondarypos

# Create pergameavg positions
rookies_transformed["position"] = [
    combine_2k_positions(primary, secondary)
    for primary, secondary
    in zip(
        rookie_primarypos,
        rookie_secondarypos
    )
]

# Predict all target features
for target in target_features:

    model = models.get(target)

    if model is not None:

        rookies_transformed[target] = model.predict(
            rookie_X
        )

# Reset index
rookies_transformed = rookies_transformed.reset_index(
    drop=True
)

# Check
print("\nTRANSFORMED ROOKIES:")

print(
    rookies_transformed[
        [
            "PlayerName",
            "primarypos",
            "secondarypos",
            "position"
        ]
    ].to_string(index=False)
)

print("\nShape:")
print(rookies_transformed.shape)


TRANSFORMED ROOKIES:
        PlayerName  primarypos  secondarypos position
   Darryn Peterson         2.0           1.0        G
       AJ Dybantsa         3.0           4.0        F
    Cameron Boozer         4.0           5.0      F-C
      Caleb Wilson         4.0           5.0      F-C
     Keaton Wagler         2.0           1.0        G
  Darius Acuff Jr.         1.0           NaN        G
     Mario Hezonja         3.0           4.0        F
   Mikel Brown Jr.         1.0           2.0        G
 Kingston Flemings         1.0           2.0        G
   Yaxel Lendeborg         4.0           3.0        F
   Brayden Burries         2.0           3.0      G-F
         Aday Mara         5.0           NaN        C
       Karim Lopez         3.0           4.0        F
      Cameron Carr         2.0           3.0      G-F
        Nate Ament         3.0           4.0        F
     Thomas Sorber         5.0           4.0      C-F
      Zuby Ejiofor         5.0           4.0      C-F
   Chr

In [30]:
# Fill rookies pergameavg for the 26_27 season
PerGameAvg["26_27"] = PerGameAvg["26_27"].copy()

rookie_lookup = rookies_transformed.set_index("PlayerName")

# Identify rookies
rookie_mask = PerGameAvg["26_27"]["PlayerName"].isin(
    rookie_lookup.index
)

# Fill model predicted target values
for target in target_features:

    if target in rookie_lookup.columns:

        PerGameAvg["26_27"].loc[
            rookie_mask,
            target
        ] = (
            PerGameAvg["26_27"]
            .loc[rookie_mask, "PlayerName"]
            .map(rookie_lookup[target])
        )

# Fill position into Pos column
PerGameAvg["26_27"].loc[
    rookie_mask,
    "Pos"
] = (
    PerGameAvg["26_27"]
    .loc[rookie_mask, "PlayerName"]
    .map(rookie_lookup["position"])
)

# Check
print(f"Rookies filled: {rookie_mask.sum()}")

print("\nFilled rookies:")

print(
    PerGameAvg["26_27"]
    .loc[
        rookie_mask,
        ["PlayerName", "Pos"] + target_features
    ]
    .to_string(index=False)
)

Rookies filled: 66

Filled rookies:
        PlayerName Pos        GP    FG_PCT   FG3_PCT    FT_PCT Restricted Area Restricted Area.1 Restricted Area.2 In The Paint (Non-RA) In The Paint (Non-RA).1 In The Paint (Non-RA).2  Mid-Range Mid-Range.1 Mid-Range.2 Left Corner 3 Left Corner 3.1 Left Corner 3.2 Right Corner 3 Right Corner 3.1 Right Corner 3.2 Above the Break 3 Above the Break 3.1 Above the Break 3.2 Backcourt Backcourt.1 Backcourt.2   Corner 3 Corner 3.1 Corner 3.2  D_FREQ      D_FGM      D_FGA  D_FG_PCT  D_NORMAL_FG_PCT  D_PCT_PLUSMINUS  FGA_FREQUENCY  EFG_PCT  FG2A_FREQUENCY       FG2M       FG2A  FG2_PCT  FG3A_FREQUENCY  PTSOFFTO   2ndPTS     FBPs     PITP  Opp PTSOFF TO  Opp2nd PTS  OppFBPs   OppPITP  RemainingGames       Min      FGM       FGA       3PM      FG3A      FTM      FTA     OReb     DReb      REB      Ast       TO      STL      BLK     BLKA       PF      PFD       Pts  NBA_FANTASY_PTS       DD2       TD3  Pts_Paint  Pts_MidRange   Pts_3PT  PLUS_MINUS
    Ugonna On

/var/folders/0b/n3f548tn3156qpndhm9hlkhh0000gn/T/ipykernel_9238/1517511970.py:16: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 4.262618   13.148291    3.748776   10.966946   11.261095    0.25771648
 17.775105   26.275908   -1.3865786  46.689255    6.9871616   0.48224273
 12.744284    1.7104288   0.47900546  6.407414    3.9851806   1.9707501
  5.881489    3.3504097  28.397736    5.2805057   0.76293486 47.798954
  4.140121   34.081177   24.029148    3.2114587   2.6571782   2.272371
  6.916379    7.515614    1.9493376   1.5273962  13.184913    7.6049366
  5.605511   17.868107    2.717323    7.367136    5.860447   12.4462385
 10.910557    4.0978413   5.6570396   7.0710254  12.356931    6.901443
  0.8389959  16.742785    2.855486   10.489775    3.5877585   1.1559951
  5.341297    4.1706443   2.5323644   4.072696    8.433491    8.486126
 -1.416979   10.267068   17.852776    4.445592   10.033653    5.378837 

Debugging

In [31]:
matches = PerGameAvg["26_27"][
    PerGameAvg["26_27"]["PlayerName"]
    .astype(str)
    .str.contains("Darius Acuff", case=False, na=False)
]

print(matches.to_string(index=False))

 PlayerID       PlayerName  TeamID TeamAbbv  Age        GP  W  L  W_Pct   FG_PCT  FG3_PCT   FT_PCT Restricted Area Restricted Area.1 Restricted Area.2 In The Paint (Non-RA) In The Paint (Non-RA).1 In The Paint (Non-RA).2 Mid-Range Mid-Range.1 Mid-Range.2 Left Corner 3 Left Corner 3.1 Left Corner 3.2 Right Corner 3 Right Corner 3.1 Right Corner 3.2 Above the Break 3 Above the Break 3.1 Above the Break 3.2 Backcourt Backcourt.1 Backcourt.2   Corner 3 Corner 3.1 Corner 3.2 Pos  D_FREQ      D_FGM      D_FGA  D_FG_PCT  D_NORMAL_FG_PCT  D_PCT_PLUSMINUS  FGA_FREQUENCY  EFG_PCT  FG2A_FREQUENCY      FG2M       FG2A  FG2_PCT  FG3A_FREQUENCY  PTSOFFTO   2ndPTS     FBPs     PITP  Opp PTSOFF TO  Opp2nd PTS  OppFBPs   OppPITP  RemainingGames       Min      FGM      FGA      3PM     FG3A     FTM    FTA     OReb     DReb      REB      Ast       TO      STL     BLK     BLKA       PF      PFD      Pts  NBA_FANTASY_PTS      DD2      TD3  Pts_Paint  Pts_MidRange  Pts_3PT  PLUS_MINUS
        0 Darius Acuff

In [32]:
PerGameAvg["26_27"] = PerGameAvg["26_27"].drop_duplicates(
    subset=["PlayerID", "PlayerName"]
)

name_overrides2 = {
    "Damian Lillard": "POR",
    "Tyrese Haliburton": "IND",
    "Kyrie Irving": "DAL",
    "Jimmy Butler": "GSW",
    "Ja Morant": "POR",
    "Trae Young": "WAS",
    "Anthony Davis": "WAS",
    "Fred VanVleet": "HOU",
    "Domantas Sabonis": "SAC",
    "Jaren Jackson Jr.": "UTA",
    "Zach LaVine": "SAC",
    "Zach Edey": "MEM",
    "Walker Kessler": "LAL"
}

PerGameAvg["26_27"].loc[
    PerGameAvg["26_27"]["PlayerName"].isin(name_overrides2),
    "TeamAbbv"
] = PerGameAvg["26_27"]["PlayerName"].map(name_overrides2)


# Step 9: Generate team rosters for each season

In [33]:
# Dicionary of the projected ESPN depth charts for each team
depth_charts = {
    "ATL": {
        "PG": ["C. McCollum", "K. Flemings", "R. Nembhard Jr.", "N. Alexander-Walker", "D. Daniels", "R. Dennis", "K. Gilbert"],
        "SG": ["N. Alexander-Walker", "L. Dort", "C. Kispert", "A. Wiggins", "C. McCollum", "D. Daniels", "B. Hield", "K. Gilbert", "J. Wilson"],
        "SF": ["D. Daniels", "A. Wiggins", "L. Dort", "C. Kispert", "B. Hield", "N. Alexander-Walker", "J. Wilson"],
        "PF": ["J. Johnson", "A. Wiggins", "M. Gueye", "A. Newell", "Z. Ejiofor", "C. Kispert", "J. Wilson"],
        "C": ["O. Okongwu", "J. Landale", "M. Gueye", "Z. Ejiofor", "H. Veesaar", "T. Bradley", "A. Newell"],
    },

    "BOS": {
        "PG": ["D. White", "P. Pritchard", "M. Conley"],
        "SG": ["B. Scheierman", "P. Pritchard", "H. Gonzalez", "D. White", "R. Harper Jr."],
        "SF": ["P. George", "S. Hauser", "B. Scheierman", "H. Gonzalez", "R. Harper Jr.", "J. Walsh", "D. Mitchell"],
        "PF": ["J. Tatum", "J. Walsh", "P. George", "C. Cenac Jr.", "S. Hauser", "R. Harper Jr.", "H. Gonzalez", "L. Garza", "B. Scheierman", "D. Mitchell"],
        "C": ["N. Queta", "M. Robinson", "L. Garza", "C. Cenac Jr.", "A. Williams"],
    },

    "BKN": {
        "PG": ["M. Brown Jr.", "B. Saraf", "N. Traore", "E. Demin", "T. Mann"],
        "SG": ["E. Demin", "D. Powell", "K. Ellis", "M. Brown Jr.", "T. Mann"],
        "SF": ["M. Porter Jr.", "K. Ellis", "N. Clowney", "J. Minott", "T. Mann", "C. Johnson", "D. Powell"],
        "PF": ["J. Randle", "N. Clowney", "J. Jefferson", "D. Wolf", "K. Ellis", "J. Minott", "C. Johnson", "T. Bilodeau"],
        "C": ["D. Sharpe", "M. Wagner", "D. Wolf", "J. Minott", "C. Johnson", "G. Nelson"],
    },

    "CHA": {
        "PG": ["C. White", "D. Schroder", "C. Anderson"],
        "SG": ["K. Knueppel", "G. Allen", "S. James", "C. White", "D. Schroder", "L. McNeeley", "P. Connaughton"],
        "SF": ["B. Miller", "S. James", "G. Allen", "R. O'Neale", "D. Finney-Smith", "L. McNeeley", "T. Salaun", "P. Connaughton"],
        "PF": ["N. Reid", "R. O'Neale", "G. Williams", "D. Finney-Smith", "T. Salaun", "S. James", "H. Steinbach", "L. McNeeley", "P. Connaughton", "P. Hall"],
        "C": ["H. Steinbach", "M. Diabate", "R. Kalkbrenner", "N. Reid", "P. Hall"],
    },

    "CHI": {
        "PG": ["J. Giddey", "T. Jones", "R. Dillingham"],
        "SG": ["N. Powell", "I. Okoro", "T. Jones", "R. Dillingham"],
        "SF": ["M. Buzelis", "D. Swain", "I. Okoro", "N. Powell", "C. Wilson", "L. Miller", "N. Essengue", "P. Williams"],
        "PF": ["C. Wilson", "P. Williams", "D. Swain", "M. Buzelis", "L. Miller", "J. Smith", "N. Essengue", "Z. Collins", "T. Awaka"],
        "C": ["N. Claxton", "J. Smith", "Z. Collins", "T. Awaka", "L. Miller"],
    },

    "CLE": {
        "PG": ["J. Harden", "C. Porter Jr.", "T. Proctor"],
        "SG": ["D. Mitchell", "S. Merrill", "T. Proctor", "J. Tyson", "C. Porter Jr.", "M. Thomas"],
        "SF": ["P. Watson", "J. Tyson", "S. Merrill", "M. Hezonja", "N. Tomlin", "T. Enaruna", "R. Minix"],
        "PF": ["E. Mobley", "P. Watson", "N. Tomlin", "J. Tyson", "M. Hezonja", "T. Enaruna", "R. Minix"],
        "C": ["J. Allen", "E. Mobley", "T. Bryant", "K. Diop", "E. Udeh Jr."],
    },

    "DAL": {
        "PG": ["K. Irving", "M. Sasser", "C. Flagg", "S. de Larrea"],
        "SG": ["M. Christie", "Z. Risacher", "C. Martin", "M. Sasser", "N. Marshall", "S. de Larrea", "J. Poulakidas", "J. Howard", "K. Irving", "V. Ishchenko", "T. Biberovic"],
        "SF": ["C. Flagg", "N. Marshall", "Z. Risacher", "S. Aldama", "C. Martin", "J. Poulakidas", "M. Christie", "P. Washington", "T. Lawal", "J. Howard", "V. Ishchenko", "T. Biberovic"],
        "PF": ["P. Washington", "M. Johnson Jr.", "S. Aldama", "C. Martin", "C. Flagg", "N. Marshall", "T. Lawal", "J. Howard", "T. Biberovic"],
        "C": ["D. Lively II", "D. Gafford", "M. Cisse", "S. Aldama", "M. Johnson Jr.", "P. Washington"],
    },

    "DEN": {
        "PG": ["J. Murray", "T. Jones", "K. Simpson"],
        "SG": ["C. Braun", "D. DeRozan", "J. Strawther", "S. Jones", "L. Walker IV", "T. Jones", "C. Johnson"],
        "SF": ["C. Johnson", "D. DeRozan", "J. Strawther", "L. Walker IV", "A. Diallo", "S. Jones", "C. Braun", "B. Hopkins"],
        "PF": ["A. Gordon", "D. DeRozan", "S. Jones", "A. Diallo", "T. Brazile", "C. Johnson", "M. Bagley III", "B. Hopkins", "Z. Nnaji", "J. Strawther"],
        "C": ["N. Jokic", "M. Bagley III", "A. Gordon", "Z. Nnaji", "D. Holmes II", "T. Brazile"],
    },

    "DET": {
        "PG": ["C. Cunningham", "D. Jenkins", "E. Okorie", "E. Harkless"],
        "SG": ["D. Robinson", "I. Joe", "K. Huerter", "G. Harris", "D. Jenkins", "R. Holland II", "T. Prince", "C. Lanier", "E. Harkless"],
        "SF": ["A. Thompson", "D. Robinson", "I. Joe", "K. Huerter", "J. Green", "T. Prince", "R. Holland II", "G. Harris", "C. Lanier"],
        "PF": ["J. Collins", "R. Holland II", "T. Prince", "I. Jones"],
        "C": ["J. Duren", "P. Reed", "J. Collins", "T. Smith", "U. Onyenso", "I. Jones"],
    },

    "GSW": {
        "PG": ["S. Curry", "D. Melton", "B. Podziemski", "B. Williams", "G. Payton II", "L. Cryer", "W. Richard"],
        "SG": ["B. Podziemski", "D. Melton", "W. Richard", "G. Payton II", "G. Santos", "M. Moody"],
        "SF": ["G. Santos", "Y. Lendeborg", "W. Richard", "G. Payton II", "L. Jones", "G. Niang", "M. Leons", "J. Butler III", "M. Moody"],
        "PF": ["D. Green", "Y. Lendeborg", "G. Santos", "G. Niang", "K. Porzingis", "A. Horford", "L. Jones", "M. Leons", "J. Butler III", "M. Moody"],
        "C": ["K. Porzingis", "A. Horford", "D. Green", "C. Bassey"],
    },

    "HOU": {
        "PG": ["F. VanVleet", "M. Smart", "A. Thompson", "B. Thornton", "R. Sheppard", "B. Bogdanovic", "Q. Copeland", "S. Pedulla"],
        "SG": ["A. Thompson", "R. Sheppard", "M. Smart", "B. Bogdanovic", "T. Eason", "K. Durant", "B. Thornton", "Q. Copeland", "S. Pedulla"],
        "SF": ["K. Durant", "T. Eason", "B. Bogdanovic", "J. Tate", "J. Smith Jr.", "I. Crawford"],
        "PF": ["J. Smith Jr.", "T. Eason", "J. Tate", "A. Sengun", "I. Crawford"],
        "C": ["A. Sengun", "S. Adams", "J. Smith Jr.", "C. Capela", "O. Tshiebwe", "R. Castro"],
    },

    "IND": {
        "PG": ["T. Haliburton", "T. McConnell", "A. Nembhard", "Q. Jackson", "B. Smith", "B. Sheppard"],
        "SG": ["A. Nembhard", "K. Oubre Jr.", "B. Sheppard", "J. Furphy", "Q. Jackson", "A. Nesmith"],
        "SF": ["A. Nesmith", "J. Walker", "K. Oubre Jr.", "J. Furphy", "K. Brown", "J. Slawson", "B. Sheppard", "Q. Jackson"],
        "PF": ["P. Siakam", "O. Toppin", "K. Oubre Jr.", "K. Brown", "L. Nance Jr.", "J. Walker", "J. Slawson", "J. Huff"],
        "C": ["I. Zubac", "L. Nance Jr.", "J. Huff", "K. Brown", "O. Toppin", "P. Siakam"],
    },

    "LAC": {
        "PG": ["D. Garland", "K. Dunn", "K. Wagler", "B. Beal", "J. Miller", "J. Pickett"],
        "SG": ["K. Wagler", "B. Beal", "G. Dick", "K. Dunn", "K. Sanders", "C. Christie", "J. Miller"],
        "SF": ["B. Ingram", "M. Strus", "K. Sanders", "R. Hachimura", "D. Jones Jr.", "G. Dick", "J. Miller", "N. Martinelli", "C. Christie"],
        "PF": ["D. Jones Jr.", "R. Hachimura", "M. Strus", "B. Miller", "K. Sanders", "J. Broome", "N. Martinelli"],
        "C": ["B. Lopez", "I. Jackson", "B. Miller", "Y. Niederhauser", "J. Broome", "J. Sharp"],
    },

    "LAL": {
        "PG": ["L. Doncic", "C. Sexton", "A. Reaves", "Q. Grimes", "B. James"],
        "SG": ["A. Reaves", "C. Sexton", "C. Carr", "M. Thybulle", "Z. Williams", "Q. Grimes", "J. LaRavia", "B. James", "J. Hardy", "D. Knecht", "C. Manon"],
        "SF": ["Q. Grimes", "C. Carr", "M. Thybulle", "Z. Williams", "D. Knecht", "A. Thiero", "J. LaRavia", "J. Hardy", "C. Manon", "A. Kaluma", "A. Okereke"],
        "PF": ["J. LaRavia", "S. Mamukelashvili", "J. Vanderbilt", "M. Thybulle", "Z. Williams", "A. Thiero", "A. Kaluma", "A. Okereke"],
        "C": ["W. Kessler", "K. Looney", "S. Mamukelashvili"],
    },

    "MEM": {
        "PG": ["S. Pippen Jr.", "T. Jerome", "C. Spencer", "J. Small", "W. Clayton Jr.", "D. Russell", "J. Mashack"],
        "SG": ["C. Coward", "T. Jerome", "W. Clayton Jr.", "C. Spencer", "J. Hawkins", "D. Russell", "J. Mashack", "J. Small", "S. Pippen Jr.", "R. Saunders"],
        "SF": ["J. Grant", "J. Wells", "G. Jackson", "K. Lopez", "C. Spencer", "C. Coward", "K. Murray", "O. Prosper", "M. Peavy", "J. Mashack", "R. Saunders"],
        "PF": ["C. Boozer", "T. Hendricks", "I. Stewart", "G. Jackson", "J. Grant", "K. Lopez", "O. Prosper", "K. Murray", "J. Wells"],
        "C": ["Z. Edey", "I. Stewart", "Q. Post", "T. Hendricks", "O. Prosper", "C. Boozer"],
    },

    "MIA": {
        "PG": ["D. Mitchell", "D. Smith", "R. Conwell", "T. Donaldson"],
        "SG": ["T. Hardaway Jr.", "K. Thompson", "R. Conwell", "P. Larsson", "M. Gardner", "D. Smith"],
        "SF": ["A. Wiggins", "K. Thompson", "P. Larsson", "T. Hardaway Jr.", "M. Gardner", "S. Fontecchio", "K. Johnson"],
        "PF": ["G. Antetokounmpo", "B. Portis", "K. Thompson", "A. Wiggins", "S. Fontecchio", "N. Jovic", "M. Gardner", "B. Adebayo", "K. Johnson"],
        "C": ["B. Adebayo", "B. Portis", "N. Richards", "N. Jovic", "V. Goldin"],
    },

    "MIL": {
        "PG": ["R. Rollins", "K. Porter Jr.", "K. Jakucionis", "C. LeVert", "B. Burries", "T. Herro", "C. Ryan", "K. Jones", "A. Green"],
        "SG": ["T. Herro", "B. Burries", "G. Trent Jr.", "C. LeVert", "K. Porter Jr.", "K. Jakucionis", "J. Jaquez Jr.", "A. Green", "C. Ryan", "K. Jones", "R. Rollins"],
        "SF": ["J. Jaquez Jr.", "O. Dieng", "G. Trent Jr.", "N. Ament", "K. Kuzma", "A. Green", "C. LeVert"],
        "PF": ["K. Kuzma", "N. Ament", "O. Dieng", "K. Ware", "J. Jaquez Jr.", "B. Markovic", "J. Sims", "P. Nance"],
        "C": ["M. Turner", "K. Ware", "J. Sims", "O. Dieng", "B. Markovic"],
    },

    "MIN": {
        "PG": ["L. Ball", "I. Evans", "B. Hyland", "A. Dosunmu", "Z. Pullin", "D. DiVincenzo"],
        "SG": ["A. Edwards", "T. Shannon Jr.", "I. Evans", "A. Dosunmu", "B. Hyland", "J. Clark", "C. Williams", "Z. Pullin"],
        "SF": ["A. Dosunmu", "J. Kuminga", "T. Shannon Jr.", "J. Clark", "C. Williams", "J. McDaniels", "I. Evans"],
        "PF": ["J. McDaniels", "J. Kuminga", "E. Freeman", "T. Shannon Jr.", "T. Kaufman-Renn", "T. Lyles", "C. Williams"],
        "C": ["R. Gobert", "J. Beringer", "R. Zikarsky", "E. Freeman", "T. Kaufman-Renn"],
    },

    "NOP": {
        "PG": ["D. Murray", "J. Fears", "J. Poole", "T. Murphy III"],
        "SG": ["T. Murphy III", "B. Mathurin", "J. Fears", "J. Pierre Jr.", "J. Poole", "H. Jones", "B. McGowens", "A. Johnson"],
        "SF": ["S. Bey", "B. Mathurin", "H. Jones", "J. Pierre Jr.", "B. McGowens"],
        "PF": ["Z. Williamson", "D. Queen", "K. Matkovic", "S. Bey", "T. Watford", "M. Dia"],
        "C": ["H. Jones", "Y. Missi", "K. Matkovic", "D. Queen", "D. Jordan", "C. Koloko", "Z. Williamson", "M. Dia"],
    },

    "NYK": {
        "PG": ["J. Brunson", "M. McBride", "J. Alvarado", "T. Kolek", "J. Clarkson", "J. Kayil"],
        "SG": ["J. Hart", "L. Shamet", "M. McBride", "J. Clarkson", "P. Dadiet", "J. Kayil", "T. Nickel"],
        "SF": ["M. Bridges", "J. Clarkson", "J. Hart", "T. Nickel", "L. Shamet", "P. Dadiet"],
        "PF": ["O. Anunoby", "K. Towns", "M. Diawara", "J. Hart", "P. Dadiet"],
        "C": ["K. Towns", "A. Drummond", "M. Diawara"],
    },

    "OKC": {
        "PG": ["S. Gilgeous-Alexander", "A. Mitchell", "B. Stirtz", "N. Topic", "J. McCain", "C. Wallace", "O. Oweh"],
        "SG": ["C. Wallace", "J. McCain", "A. Mitchell", "B. Stirtz", "A. Caruso", "N. Topic", "O. Oweh", "J. Dix"],
        "SF": ["J. Williams", "C. Wallace", "A. Caruso", "K. Williams", "B. Barnhizer", "O. Oweh", "J. Dix"],
        "PF": ["C. Holmgren", "A. Caruso", "K. Williams", "J. Williams", "B. Barnhizer"],
        "C": ["I. Hartenstein", "J. Williams", "A. Mara", "T. Sorber"],
    },

    "ORL": {
        "PG": ["J. Suggs", "A. Black", "J. Carter", "J. Richardson", "A. Morales"],
        "SG": ["D. Bane", "A. Black", "J. Cain", "N. Penda", "J. Richardson", "J. Carter", "A. Morales"],
        "SF": ["F. Wagner", "J. Cain", "T. da Silva", "N. Penda", "D. Bane"],
        "PF": ["P. Banchero", "T. da Silva", "N. Penda", "J. Cain", "J. Isaac", "I. Nelson", "C. Castleton"],
        "C": ["W. Carter Jr.", "N. Vucevic", "G. Bitadze", "I. Nelson", "C. Castleton"],
    },

    "PHI": {
        "PG": ["T. Maxey", "L. Philon Jr.", "A. Simons", "V. Edgecombe", "K. Caldwell-Pope", "C. Love"],
        "SG": ["V. Edgecombe", "A. Simons", "K. Caldwell-Pope", "L. Philon Jr.", "J. Brown", "R. Rupert", "C. Love"],
        "SF": ["J. Brown", "K. Caldwell-Pope", "J. Edwards", "J. Walker", "D. Wade", "L. James", "R. Rupert"],
        "PF": ["L. James", "D. Wade", "D. Barlow", "J. Edwards", "J. Walker", "A. Bona", "R. Rupert"],
        "C": ["J. Embiid", "A. Bona", "A. Hukporti", "D. Barlow", "J. Walker"],
    },

    "PHX": {
        "PG": ["D. Booker", "C. Gillespie", "L. Kennard", "J. Bouyea", "J. Goodwin", "P. Spencer"],
        "SG": ["J. Green", "L. Kennard", "D. Booker", "J. Goodwin", "C. Gillespie", "J. Bouyea", "P. Spencer", "K. Brea"],
        "SF": ["D. Brooks", "J. Goodwin", "K. Peat", "R. Fleming", "R. Dunn", "L. Kennard", "J. Green", "H. Highsmith", "K. Brea"],
        "PF": ["M. Bridges", "R. Dunn", "K. Peat", "R. Fleming", "D. Brooks", "J. Goodwin", "H. Highsmith", "C. Huntley"],
        "C": ["O. Ighodaro", "K. Maluach", "K. Peat", "R. Fleming", "C. Huntley", "M. Williams"],
    },

    "POR": {
        "PG": ["J. Morant", "S. Henderson", "J. Holiday", "B. Wesley", "D. Avdija", "V. Krejci", "D. Lillard", "S. Sharpe"],
        "SG": ["D. Lillard", "J. Holiday", "V. Krejci", "J. Morant", "T. Camara", "B. Wesley", "S. Sharpe", "C. Youngblood", "J. Tonje"],
        "SF": ["T. Camara", "V. Krejci", "J. Sochan", "J. Holiday", "D. Avdija", "S. Cissoko", "S. Sharpe", "J. Kent", "J. Tonje"],
        "PF": ["D. Avdija", "S. Cissoko", "J. Sochan", "M. Potter", "R. Williams III", "J. Kent"],
        "C": ["D. Clingan", "R. Williams III", "M. Potter", "S. Cissoko", "Y. Hansen", "B. Carlson"],
    },

    "SAC": {
        "PG": ["D. Acuff Jr.", "M. Monk", "B. Simmons", "A. Flagler"],
        "SG": ["Z. LaVine", "N. Clifford", "M. Monk", "E. Sharp", "D. Plowden", "A. Flagler"],
        "SF": ["D. Hunter", "N. Clifford", "B. Simmons", "A. Karaban", "M. Monk", "D. Plowden", "K. Murray"],
        "PF": ["K. Murray", "P. Achiuwa", "D. Hunter", "D. Cardwell", "A. Karaban", "D. Plowden", "D. Sabonis", "N. Clifford", "J. Mogbo"],
        "C": ["D. Sabonis", "M. Raynaud", "D. Cardwell", "P. Achiuwa", "J. Mogbo"],
    },

    "SAS": {
        "PG": ["D. Fox", "D. Harper", "J. McLaughlin", "J. Gillespie", "D. Jones Garcia"],
        "SG": ["S. Castle", "D. Harper", "J. Champagnie", "J. McLaughlin", "K. Johnson", "D. Jones Garcia"],
        "SF": ["D. Vassell", "C. Bryant", "K. Johnson", "J. Champagnie", "H. Barnes", "D. Jones Garcia"],
        "PF": ["T. Harris", "J. Champagnie", "K. Johnson", "H. Barnes", "C. Bryant", "J. Quaintance", "L. Kornet", "M. Brown"],
        "C": ["V. Wembanyama", "L. Kornet", "T. Reed Jr.", "J. Quaintance"],
    },

    "TOR": {
        "PG": ["I. Quickley", "J. Shead", "J. Walter", "J. Bradley", "S. Barnes", "C. Hepburn", "M. Smith"],
        "SG": ["R. Barrett", "J. Walter", "J. Shead", "J. Bradley", "A. Martin", "M. Smith"],
        "SF": ["K. Leonard", "A. Graves", "J. Battle", "S. Barnes", "K. Anderson", "J. Walter"],
        "PF": ["S. Barnes", "C. Murray-Boyles", "K. Anderson", "A. Graves", "R. Barrett", "J. Battle"],
        "C": ["J. Poeltl", "C. Murray-Boyles", "T. Jackson-Davis", "J. Battle", "T. Jemison III"],
    },

    "UTA": {
        "PG": ["K. George", "I. Collier", "D. Peterson", "T. Alexander"],
        "SG": ["D. Peterson", "A. Bailey", "J. Okogie", "B. Sensabaugh", "S. Mykhailiuk", "T. Bates", "T. Alexander", "J. Green"],
        "SF": ["L. Markkanen", "B. Sensabaugh", "J. Okogie", "J. Green", "A. Bailey", "S. Mykhailiuk", "H. Ingram", "B. Hinson"],
        "PF": ["J. Jackson Jr.", "K. Filipowski", "A. Bailey", "L. Markkanen", "B. Hinson", "B. Sensabaugh", "H. Ingram", "J. Okogie"],
        "C": ["J. Nurkic", "J. Hayes", "K. Filipowski", "J. Jackson Jr.", "M. Bamba", "B. Hinson"],
    },

    "WAS": {
        "PG": ["T. Young", "B. Carrington", "T. Mann", "T. Johnson"],
        "SG": ["K. George", "T. Johnson", "B. Carrington", "W. Riley", "A. Williams", "T. Mann", "J. Watkins"],
        "SF": ["A. Dybantsa", "B. Coulibaly", "W. Riley", "K. Middleton", "K. George", "J. Champagnie", "T. Johnson", "J. Watkins"],
        "PF": ["A. Davis", "K. Middleton", "J. Champagnie", "W. Riley", "A. Dybantsa", "B. Coulibaly", "T. Vukcevic", "J. Watkins"],
        "C": ["A. Sarr", "D. Ayton", "T. Vukcevic", "A. Davis", "F. Okpara", "J. Champagnie"],
    },
}

In [34]:
# Helper function that creates team rosters
def create_team_rosters(data):

    team_rosters = {}

    for season, df in data.items():

        if "TeamAbbv" not in df.columns:
            raise ValueError(
                f"{season} is missing 'TeamAbbv'"
            )

        team_rosters[season] = {}

        teams = sorted(
            df["TeamAbbv"]
            .dropna()
            .unique()
        )

        for team in teams:

            team_rosters[season][team] = df[
                df["TeamAbbv"] == team
            ].copy()

    return team_rosters


In [35]:
# Helper function that Matches depth chart to rosters
def match_depth_charts_to_rosters(
    depth_charts,
    team_rosters,
    season="26_27"
):

    matched_depth_charts = {}
    unmatched_players = {}

    for team_abbr, positions in depth_charts.items():

        # Check that team exists
        if season not in team_rosters:
            raise ValueError(
                f"Season '{season}' not found in team_rosters"
            )

        if team_abbr not in team_rosters[season]:
            print(
                f"WARNING: {team_abbr} not found in "
                f"team_rosters[{season}]"
            )
            continue

        # Get roster
        roster = team_rosters[season][team_abbr].copy()

        # Create last name column
        roster["_LastName"] = (
            roster["PlayerName"]
            .astype(str)
            .str.strip()
            .str.split()
            .str[-1]
            .str.lower()
        )

        matched_depth_charts[team_abbr] = {}
        unmatched_players[team_abbr] = []

        # Loop through positions
        for position, players in positions.items():

            matched_depth_charts[team_abbr][position] = []

            # Loop through depth chart players
            for depth_player in players:

                # Example:
                # "A. Davis" -> "davis"

                depth_last_name = (
                    depth_player
                    .strip()
                    .split()[-1]
                    .lower()
                )

                # Find matching player(s)
                matches = roster[
                    roster["_LastName"] == depth_last_name
                ]

                # Match found
                if not matches.empty:

                    for _, row in matches.iterrows():

                        matched_depth_charts[
                            team_abbr
                        ][position].append(
                            row["PlayerName"]
                        )

                # No match
                else:

                    unmatched_players[
                        team_abbr
                    ].append(
                        depth_player
                    )

        # Remove temporary column
        roster.drop(
            columns="_LastName",
            inplace=True
        )

    return matched_depth_charts, unmatched_players

In [36]:
# Create team rosters from per game data
team_rosters = create_team_rosters(PerGameAvg)


# Match depth chart to rosters
matched_depth_charts, unmatched_players = (
    match_depth_charts_to_rosters(
        depth_charts,
        team_rosters,
        season="26_27"
    )
)

In [37]:
# Helper function that gets last name
def get_last_name(name):

    suffixes = {
        "jr",
        "jr.",
        "sr",
        "sr.",
        "ii",
        "iii",
        "iv",
        "v"
    }

    parts = (
        str(name)
        .strip()
        .replace(",", "")
        .split()
    )

    if not parts:
        return ""

    # Remove suffix
    if parts[-1].lower() in suffixes and len(parts) > 1:
        parts = parts[:-1]

    return parts[-1].lower()

In [38]:
matched_depth_charts, unmatched_players = match_depth_charts_to_rosters(
    depth_charts,
    team_rosters,
    season="26_27"
)

In [39]:
# Helper function that creates a dictionary of NBA team rosters from player data.
# Expected structure: data[season] = DataFrame
# Required columns: TeamAbbv
# Returns: rosters[season][team_abbreviation] = DataFrame
def create_team_rosters(data):
    team_rosters = {}

    for season, df in data.items():

        if "TeamAbbv" not in df.columns:
            raise ValueError(
                f"{season} is missing 'TeamAbbv'"
            )

        team_rosters[season] = {}

        teams = sorted(
            df["TeamAbbv"]
            .dropna()
            .unique()
        )

        for team in teams:

            team_rosters[season][team] = df[
                df["TeamAbbv"] == team
            ]
            
    return team_rosters


In [40]:
team_rosters = create_team_rosters(PerGameAvg)

# Step 10: Generate 9 Man Rotation

In [41]:
# Helper function that calculates TO%
# Formula:
#     TO% = 100 * TOV / (FGA + 0.44 * FTA + TOV)

# Required columns:
#     FGA - Field goal attempts
#     FTA - Free throw attempts
#     TO  - Turnovers

# Returns:
#     pandas.Series containing TO%
def calculate_team_to_pct(df):
    denominator = df["FGA"] + 0.44 * df["FTA"] + df["TO"]

    return 100 * df["TO"] / denominator


In [42]:
# Helper function that Calculates the percentage of total points scored in the paint.
# Formula:
#     %PTS PITP = (Points in Paint / Total Points) * 100
# Required columns:
#     Pts_Paint - Points scored in the paint
#     Pts       - Total points scored
# Returns:
#     pandas.Series containing %PTS PITP
def calculate_pct_points_in_paint(df):
    return (df["Pts_Paint"] / df["Pts"]) * 100


In [43]:
# Helper function that creates a ghost player
stat_columns = [
    "FGM",
    "FGA",
    "3PM",
    "FG3A",
    "FTM",
    "FTA",
    "OReb",
    "DReb",
    "REB",
    "Ast",
    "TO",
    "STL",
    "BLK",
    "BLKA",
    "PF",
    "PFD",
    "Pts",
    "NBA_FANTASY_PTS",
    "DD2",
    "TD3",
    "Pts_Paint",
    "Pts_MidRange",
    "Pts_3PT",
    "PLUS_MINUS",
    "PTSOFFTO",
    "2ndPTS",
    "FBPs",
    "PITP",
    "Opp PTSOFF TO",
    "Opp2nd PTS",
    "OppFBPs",
    "OppPITP"
]

In [44]:
# Helper function that gets the 9 man rotation of a team
def get_9_man_rotation(team_df):
    
    # Only available players
    available = team_df[
        team_df["RemainingGames"] > 0
    ].copy()

    print(f"Available players: {len(available)}")

    if available.empty:
        print("No available players.")
        return available

    available["Min"] = pd.to_numeric(
        available["Min"],
        errors="coerce"
    ).fillna(0)
    
    # Helper functions to identify positions
    def is_center(pos):
        return "C" in str(pos).upper()

    def is_guard(pos):
        return "G" in str(pos).upper()

    def is_forward(pos):
        return "F" in str(pos).upper()

    # Helper function that counts the number of players at each position
    def count_positions(df):

        centers = df["Pos"].apply(is_center).sum()
        guards = df["Pos"].apply(is_guard).sum()
        forwards = df["Pos"].apply(is_forward).sum()

        return centers, guards, forwards

    # Helper function that gets position minutes
    def position_minutes(df):

        center_minutes = df.loc[
            df["Pos"].apply(is_center),
            "Min"
        ].sum()

        guard_minutes = df.loc[
            df["Pos"].apply(is_guard),
            "Min"
        ].sum()

        forward_minutes = df.loc[
            df["Pos"].apply(is_forward),
            "Min"
        ].sum()

        return (
            center_minutes,
            guard_minutes,
            forward_minutes
        )

    # Ghost source players
    ghost_source = PerGameAvg["26_27"].copy()

    # Helper function to create ghost players
    def get_ghost_template(player_name):

        matches = ghost_source[
            ghost_source["PlayerName"]
            .astype(str)
            .str.contains(
                player_name,
                case=False,
                na=False
            )
        ]

        if matches.empty:
            raise ValueError(
                f"Could not find {player_name} "
                f"in PerGameAvg['26_27']"
            )

        return matches.iloc[0].copy()

    dario_template = get_ghost_template(
        "Dario Saric"
    )

    thanasis_template = get_ghost_template(
        "Thanasis Antetokounmpo"
    )

    lowry_template = get_ghost_template(
        "Kyle Lowry"
    )

    # Helper function to create ghost player
    def create_ghost_player(position, ghost_minutes):

        if position == "C":

            source = dario_template.copy()

        elif position == "F":

            source = thanasis_template.copy()

        elif position == "G":

            source = lowry_template.copy()

        else:

            raise ValueError(
                f"Unknown ghost position: {position}"
            )

        # Source minutes
        source_minutes = pd.to_numeric(
            source["Min"],
            errors="coerce"
        )

        if pd.isna(source_minutes) or source_minutes <= 0:

            raise ValueError(
                f"Invalid source minutes for "
                f"{source['PlayerName']}: "
                f"{source['Min']}"
            )

        ghost_minutes = float(
            ghost_minutes
        )

        # Minute scailing factor
        scale = (
            ghost_minutes /
            source_minutes
        )
        
        # Copy source player
        ghost = source.copy()

        # Identity 
        ghost_ids = {
            "C": -1001,
            "F": -1002,
            "G": -1003
        }

        ghost["PlayerID"] = ghost_ids[position]
        ghost["PlayerName"] = "Jon Snow"

        # Team
        ghost["TeamID"] = (
            team_df["TeamID"].iloc[0]
        )

        ghost["TeamAbbv"] = (
            team_df["TeamAbbv"].iloc[0]
        )

        # Position
        ghost["Pos"] = position

        # Availability
        ghost["GP"] = 0
        ghost["RemainingGames"] = 1

        # Minutes
        ghost["Min"] = ghost_minutes

        # Scale minute depending features
        for stat in stat_columns:

            if stat not in ghost.index:
                continue

            value = pd.to_numeric(
                ghost[stat],
                errors="coerce"
            )

            if pd.notna(value):

                ghost[stat] = (
                    value * scale
                )

        return ghost
    
    # Helper function that adds a ghost player for position
    def add_ghost_for_position(rotation, position):

        (
            center_minutes,
            guard_minutes,
            forward_minutes
        ) = position_minutes(rotation)
        
        # Determine position minute defecits
        if position == "C":

            minutes_needed = max(
                0,
                48.0 - center_minutes
            )

        elif position == "G":

            minutes_needed = max(
                0,
                48.0 - guard_minutes
            )

        elif position == "F":

            minutes_needed = max(
                0,
                48.0 - forward_minutes
            )

        else:

            raise ValueError(
                f"Invalid position: {position}"
            )

        # If no defecit, give small minimum
        if minutes_needed <= 0:

            minutes_needed = 1.0

        # Create ghost
        ghost = create_ghost_player(
            position,
            minutes_needed
        )

        # Find player to remove
        candidates = rotation.copy()

        if position == "C":

            non_centers = candidates[
                ~candidates["Pos"].apply(
                    is_center
                )
            ]

            if not non_centers.empty:

                candidates = non_centers

        elif position == "G":

            non_guards = candidates[
                ~candidates["Pos"].apply(
                    is_guard
                )
            ]

            if not non_guards.empty:

                candidates = non_guards

        elif position == "F":

            non_forwards = candidates[
                ~candidates["Pos"].apply(
                    is_forward
                )
            ]

            if not non_forwards.empty:

                candidates = non_forwards

        # Remove lowest minute expendable players
        remove_idx = candidates[
            "Min"
        ].idxmin()

        removed_name = rotation.loc[
            remove_idx,
            "PlayerName"
        ]

        removed_min = rotation.loc[
            remove_idx,
            "Min"
        ]

        print(
            f"  -> Replacing {removed_name} "
            f"({removed_min:.2f} min) "
            f"with Jon Snow ({position}) "
            f"({minutes_needed:.2f} min)"
        )

        rotation = rotation.drop(
            remove_idx
        )

        rotation = pd.concat(
            [
                rotation,
                ghost.to_frame().T
            ],
            ignore_index=True
        )

        return rotation

    # Sort real player by minutes
    available = available.sort_values(
        "Min",
        ascending=False
    ).reset_index(drop=True)

    # Start wit the top 9 real players
    rotation = available.head(9).copy()

    # Helper function that returns the available real players that arent already in the rotation
    def get_unused_real_players(rotation):

        used_ids = set(rotation["PlayerID"])

        return available[
            ~available["PlayerID"].isin(used_ids)
        ].copy()

# Helper function that returns the highest minute available REAL PLAYER who can play the requiested
#     position and is not already in the rotation, returns non if no real player exists
    def find_real_player_for_position(rotation, position):

        unused = get_unused_real_players(rotation)

        if position == "C":
            candidates = unused[
                unused["Pos"].apply(is_center)
            ]

        elif position == "G":
            candidates = unused[
                unused["Pos"].apply(is_guard)
            ]

        elif position == "F":
            candidates = unused[
                unused["Pos"].apply(is_forward)
            ]

        else:
            raise ValueError(
                f"Invalid position: {position}"
            )

        if candidates.empty:
            return None

        return candidates.sort_values(
            "Min",
            ascending=False
        ).iloc[0]

# Helper function that adds a real player to the rotation
    def add_real_player(rotation, player):
        return pd.concat(
            [
                rotation,
                player.to_frame().T
            ],
            ignore_index=True
        )
    
    # Position requirements
    # We want:
    #     C >= 2
    #     G >= 1
    #     F >= 1
    #
    # But we NEVER create a ghost if a real available
    # roster player can fill the position.
    required_positions = [
        ("C", 2),
        ("G", 1),
        ("F", 1)
    ]

    # Fix position requirements using real players first
    for position, required_count in required_positions:

        current_count = (
            rotation["Pos"]
            .apply(
                {
                    "C": is_center,
                    "G": is_guard,
                    "F": is_forward
                }[position]
            )
            .sum()
        )

        while current_count < required_count:

            # Look for a real available player first
            real_player = find_real_player_for_position(
                rotation,
                position
            )

            if real_player is not None:

                print(
                    f"  -> Adding real player "
                    f"{real_player['PlayerName']} "
                    f"({real_player['Pos']}, "
                    f"{real_player['Min']:.2f} min) "
                    f"to fill {position}"
                )

                # If rotation already has 9 players, remove
                # the lowest-minute player who does NOT play
                # the required position.
                if len(rotation) >= 9:

                    if position == "C":
                        removable = rotation[
                            ~rotation["Pos"].apply(is_center)
                        ]

                    elif position == "G":
                        removable = rotation[
                            ~rotation["Pos"].apply(is_guard)
                        ]

                    else:  # F
                        removable = rotation[
                            ~rotation["Pos"].apply(is_forward)
                        ]

                    if removable.empty:
                        removable = rotation

                    remove_idx = removable["Min"].idxmin()

                    removed_name = rotation.loc[
                        remove_idx,
                        "PlayerName"
                    ]

                    print(
                        f"     Removing {removed_name} "
                        f"to make room."
                    )

                    rotation = rotation.drop(
                        remove_idx
                    ).reset_index(drop=True)

                rotation = add_real_player(
                    rotation,
                    real_player
                )

            else:
                # If no real players exist create fake player Jon Snow
                print(
                    f"  -> No available real {position} "
                    f"player exists."
                )

                # Calculate current minutes at position
                if position == "C":
                    current_minutes = rotation.loc[
                        rotation["Pos"].apply(is_center),
                        "Min"
                    ].sum()

                elif position == "G":
                    current_minutes = rotation.loc[
                        rotation["Pos"].apply(is_guard),
                        "Min"
                    ].sum()

                else:
                    current_minutes = rotation.loc[
                        rotation["Pos"].apply(is_forward),
                        "Min"
                    ].sum()

                minutes_needed = max(
                    1.0,
                    48.0 - current_minutes
                )

                ghost = create_ghost_player(
                    position,
                    minutes_needed
                )

                print(
                    f"     -> Creating Jon Snow "
                    f"({position}, "
                    f"{minutes_needed:.2f} min)"
                )

                rotation = pd.concat(
                    [
                        rotation,
                        ghost.to_frame().T
                    ],
                    ignore_index=True
                )

            current_count = (
                rotation["Pos"]
                .apply(
                    {
                        "C": is_center,
                        "G": is_guard,
                        "F": is_forward
                    }[position]
                )
                .sum()
            )

    # Make sure we have 9 players
    # If we have fewer than 9, fill with REAL AVAILABLE
    # players first.
    #
    # Ghosts are only used if there are literally no more
    # real available players.
    while len(rotation) < 9:

        real_players = get_unused_real_players(rotation)

        if not real_players.empty:

            player = real_players.sort_values(
                "Min",
                ascending=False
            ).iloc[0]

            print(
                f"  -> Adding real player "
                f"{player['PlayerName']} "
                f"({player['Pos']}, "
                f"{player['Min']:.2f} min)"
            )

            rotation = add_real_player(
                rotation,
                player
            )

        else:

            # Absolute last resort if there are no real players left
            print(
                "  -> No unused real players remain. "
                "Creating Jon Snow."
            )

            # Determine the position that is currently
            # most deficient in minutes.

            center_minutes, guard_minutes, forward_minutes = (
                position_minutes(rotation)
            )

            deficits = {
                "C": max(0, 48.0 - center_minutes),
                "G": max(0, 48.0 - guard_minutes),
                "F": max(0, 48.0 - forward_minutes)
            }

            position = max(
                deficits,
                key=deficits.get
            )

            minutes_needed = max(
                1.0,
                deficits[position]
            )

            ghost = create_ghost_player(
                position,
                minutes_needed
            )

            rotation = pd.concat(
                [
                    rotation,
                    ghost.to_frame().T
                ],
                ignore_index=True
            )

    # Final Sort
    rotation["Min"] = pd.to_numeric(
        rotation["Min"],
        errors="coerce"
    ).fillna(0)

    rotation = rotation.sort_values(
        "Min",
        ascending=False
    ).reset_index(drop=True)

    # Assign starter / bench players
    rotation["Rotation_Role"] = [
        "Starter" if i < 5
        else "Bench"
        for i in range(len(rotation))
    ]

    # Final position counts
    centers, guards, forwards = count_positions(
        rotation
    )

    (
        center_minutes,
        guard_minutes,
        forward_minutes
    ) = position_minutes(rotation)

    # Final print
    return rotation

In [45]:
# Manual rotation fixes to player
stat_columns = [
    "FGM",
    "FGA",
    "3PM",
    "FG3A",
    "FTM",
    "FTA",
    "OReb",
    "DReb",
    "REB",
    "Ast",
    "TO",
    "STL",
    "BLK",
    "BLKA",
    "PF",
    "PFD",
    "Pts",
    "NBA_FANTASY_PTS",
    "DD2",
    "TD3",
    "Pts_Paint",
    "Pts_MidRange",
    "Pts_3PT",
    "PLUS_MINUS",
    "PTSOFFTO",
    "2ndPTS",
    "FBPs",
    "PITP",
    "Opp PTSOFF TO",
    "Opp2nd PTS",
    "OppFBPs",
    "OppPITP"
]

# Helper function that normalizes one player's minutes + stats
def normalize_player_downward(
    player_row,
    target_minutes=None,
    target_gp=None
):

    player = player_row.copy()
    
    # Override minutes
    if target_minutes is not None:

        original_minutes = pd.to_numeric(
            player["Min"],
            errors="coerce"
        )

        if pd.isna(original_minutes) or original_minutes <= 0:

            print(
                f"WARNING: Invalid original minutes for "
                f"{player['PlayerName']}"
            )

            return player

        target_minutes = float(target_minutes)

        # Scailing factor
        scale = (
            target_minutes /
            original_minutes
        )

        # Minutes
        player["Min"] = target_minutes

        # Scale stats
        for stat in stat_columns:

            if stat not in player.index:
                continue

            value = pd.to_numeric(
                player[stat],
                errors="coerce"
            )

            if pd.notna(value):

                player[stat] = (
                    value * scale
                )

    # Override GP 
    if target_gp is not None:

        player["GP"] = target_gp

        player["RemainingGames"] = target_gp

    return player


# Helper funciton that applies manual fixes to Specific Teams
# Rules:
# - minutes only:
#     Changes minutes and scales stats.
# - gp only:
#     Changes GP + RemainingGames.
#     Does NOT change minutes or stats.
# - minutes + gp:
#     Changes both.
# - No players are removed.
# - Only the specified team is searched.
# - Only the specified player is modified.
def apply_manual_rotation_fixes(team_rosters, season, fixes):

    # Loop through teams
    for team, player_fixes in fixes.items():

        # Check team exists
        if team not in team_rosters[season]:

            print(
                f"\nWARNING: {team} not found "
                f"in {season}"
            )

            continue

        roster = team_rosters[season][team]

        # Loop through players
        for player_name, settings in player_fixes.items():

            # Get optional seetings
            target_minutes = settings.get(
                "minutes",
                None
            )

            target_gp = settings.get(
                "gp",
                None
            )

            # Find player in the team
            matches = roster[
                roster["PlayerName"]
                .astype(str)
                .str.strip()
                .str.lower()
                ==
                player_name.strip().lower()
            ]

            if matches.empty:

                print(
                    f"\nWARNING: {player_name} "
                    f"not found on {team}"
                )

                continue

            # Get exact row

            idx = matches.index[0]

            # Before values
            old_min = pd.to_numeric(
                roster.loc[idx, "Min"],
                errors="coerce"
            )

            old_gp = pd.to_numeric(
                roster.loc[idx, "GP"],
                errors="coerce"
            )

            old_remaining = pd.to_numeric(
                roster.loc[
                    idx,
                    "RemainingGames"
                ],
                errors="coerce"
            )

            # Normalize
            normalized = normalize_player_downward(
                roster.loc[idx],
                target_minutes=target_minutes,
                target_gp=target_gp
            )

            # Write back to same row
            for column in roster.columns:
                roster.loc[idx, column] = normalized[column]
    return team_rosters

In [46]:
# Manual fixes done
manual_fixes = {

    "POR": {
        "Ja Morant": {
            "minutes": 25,
            "gp": 30
        },
        "Shaedon Sharpe": {
            "minutes": 26,
            "gp": 50
        },
        "Damian Lillard": {
            "minutes": 30,
            "gp": 40
        },
        "Deni Avdija": {
            "minutes": 30,
            "gp": 40
        },
        "Toumani Camara": {
            "minutes": 27,
            "gp": 60
        },
        "Yang Hansen": {
            "minutes": 14
        },
        "Scoot Henderson": {
            "minutes": 17,
            "gp": 60
        }
    },

    "SAC": {
        "Darius Acuff Jr.": {
            "minutes": 28,
            "gp": 50
        },
        "Zach Lavine": {
            "minutes": 28,
            "gp": 30
        },
        "Domantas Sabonis": {
            "minutes": 28,
            "gp": 30
        },
        "Keegan Murray": {
            "minutes": 28,
            "gp": 30
        }
    },

    "ATL": {
        "Luguentz Dort": {
            "minutes": 28
        },
        "Dyson Daniels": {
            "minutes": 29
        },
        "Kingston Fleming": {
            "minutes": 22
        }
    },

    "BKN": {
        "Egor Dëmin": {
            "minutes": 30,
            "gp": 60
        }
    },


    "CHI": {
        "Josh Giddey": {
            "gp": 60
        },
        "Norman Powell": {
            "gp": 65
        },
    },

    "BOS": {
        "Mitchell Robinson": {
            "minutes": 25
        },
        "Jayson Tatum": {
            "gp": 60
        },
        "Paul George 65": {
            "gp": 60
        }
    },

    "CHA": {
        "Coby White": {
            "minutes": 28
        },
        "Bez Mbeng": {
            "minutes": 20,
            "gp": 40
        },

        "Grayson Allen": {
            "minutes": 22,
            "gp": 40
        },

        "Royce O'Neale	": {
            "minutes": 21,
            "gp": 40
        }
    },

    "CLE": {
        "Peyton Watson": {
            "gp": 61
        }
    },

    "DAL": {
        "Kyrie Irving": {
            "minutes": 33,
            "gp": 65
        },
        "Zaccharie Risacher": {
            "minutes": 25,
            "gp": 62
        },
        "Daniel Gafford": {
            "minutes": 25,
            "gp": 62
        },
        "Dereck Lively II": {
            "minutes": 25,
            "gp": 57
        },
        "Naji Marshall	": {
            "minutes": 26,
            "gp": 57
        },
        "Max Christie": {
            "minutes": 25,
            "gp": 57
        },
        "Santi Aldama": {
            "minutes": 22,
            "gp": 57
        }
    },

    "DEN": {
        "Nikola Jokić": {
            "gp": 71
        },
        "Aaron Gordon": {
            "minutes": 29,
            "gp": 64
        },
        "Cameron Johnson": {
            "gp": 59
        },
        "Lonnie Walker IV": {
            "gp": 49
        },
        "Spencer Jones": {
            "minutes": 24
        },
        "Julian Strawther": {
            "minutes": 22
        }
    },

    "DET": {
        "Cade Cunningham": {
            "gp": 75
        },
        "Taurean Prince": {
            "gp": 63
        },
        "Isaiah Joe": {
            "minutes": 23
        },
        "Ronald Holland II": {
            "minutes": 25
        },
        "Paul Reed": {
            "minutes": 25
        }
    },

    "GSW": {
        "Stephen Curry": {
            "minutes": 28,
            "gp": 55
        },
        "Moses Moody": {
            "gp": 0
        },
        "Jimmy Butler II": {
            "gp": 0
        },
        "Kristaps Porziņģis": {
            "gp": 45
        },
        "De'Anthony Melton": {
            "gp": 35
        },
        "Brandon Williams": {
            "minutes": 26,
            "gp": 50
        },
        "Al Horford": {
            "gp": 40
        },
        "Yaxel Lendeborg": {
            "minutes": 25,
            "gp": 40
        },
        "Georges Niang": {
            "gp": 0,
            "minutes": 0
        }

    },


    "IND": {
        "Pascal Siakam": {
            "gp": 73
        },
        "Kelly Oubre Jr.": {
            "gp": 64
        },
        "Andrew Nembhard": {
            "gp": 64
        },
        "Ivica Zubac": {
            "gp": 68
        },
        "Aaron Nesmith": {
            "gp": 65
        },
        "Johnny Furphy": {
            "gp": 40
        },
        "Obi Toppin": {
            "gp": 40
        },
    },
    
    "LAC": {
        "Darius Garland": {
            "gp": 71
        },
        "Max Strus": {
            "gp": 75
        },
        "Bradley Beal": {
            "minutes": 28,
            "gp": 65
        }
    },

    "LAL": {
        "Luka Dončić": {
            "gp": 60
        },
        "Austin Reaves": {
            "gp": 60
        },
        "Walker Kessler": {
            "gp": 55
        },
        "Matisse Thybulle": {
            "gp": 657
        },
        "Kevon Looney": {
            "minutes": 20,
            "gp": 40
        }
    },

    "MEM": {
        "Cameron Boozer": {
            "minutes": 28,
            "gp": 60
        },
        "Jerami Grant": {
            "gp": 71
        },
        "Cedric Coward": {
            "minutes": 30,
            "gp": 65
        },
        "Ty Jerome": {
            "gp": 50
        }
    },

    "MIA": {
        "Giannis Antetokounmpo": {
            "minutes": 33,
            "gp": 70
        }
    },

    "MIL": {
        "Kevin Porter Jr.": {
            "gp": 65
        },
        "Kel'el Ware": {
            "minutes": 25
        }
    },

    "MIN": {
        "LaMelo Ball": {
            "minutes": 33,
            "gp": 75
        },
    
        "Terrence Shannon Jr": {
            "gp": 65
        },
        "Jonathan Kuminga": {
            "minutes": 30,
            "gp": 70
        },
        "Donte DiVincenzo": {
            "minutes": 30,
            "gp": 5
        },
        "Bones Hyland": {
            "minutes": 21,
        },
        "Trey Lyles": {
            "gp": 0
        },
        "Rocco Zikarsky": {
            "gp": 30,
            "minutes": 12,
        }
    },

    "NOP": {
        "Trey Murphy III": {
            "gp": 65
        },
        "Zion Williamson": {
            "gp": 60
        },
        "Bennedict Mathurin": {
            "gp": 55
        },
        "Jordan Poole": {
            "gp": 55
        },
        "Miles McBride": {
            "gp": 65
        },
        "Landry Shamet": {
            "gp": 65
        },
        "Saddiq Bey": {
            "gp": 65
        }
    },

    "HOU": {
        "Steven Adams": {
            "gp": 60
        },
        "Bogdan Bogdanović": {
            "gp": 50
        },
    },

    "OKC": {
        "Shai Gilgeous-Alexander": {
            "gp": 72
        },
        "Jalen Williams": {
            "gp": 70
        },
        "Isaiah Hartenstein": {
            "gp": 65
        },
        "Ajay Mitchell": {
            "gp": 65
        },
        "Chet Holmgren": {
            "gp": 71
        },
        "Alex Caruso": {
            "minutes": 22,
            "gp": 65
        },
        "Jared McCain": {
            "minutes": 25,
            "gp": 66
        },
        "Kenrich Williams": {
            "gp": 65
        },
        "Aday Mara": {
            "gp": 65
        },
        "Jaylin Williams": {
            "minutes": 24
        }
    },

    "ORL": {
        "Franz Wagner": {
            "gp": 65
        },
        "Nikola Vučević": {
            "gp": 65
        },
        "Jalen Suggs": {
            "gp": 69
        },
        "Goga Bitadze": {
            "gp": 65
        },
    },

    "PHI": {
        "Joel Embiid": {
            "gp": 65
        },
        "Anfernee Simons": {
            "gp": 60
        }
    },

    "PHX": {
        "Devin Booker": {
            "gp": 75
        },
        "Dillon Brooks": {
            "gp": 65
        },
        "Jalen Green": {
            "gp": 65
        },
        "Mark Williams": {
            "minutes": 28,
            "gp": 65
        },

        "Khaman Maluach": {
            "minutes": 28,
            "gp": 65
        }
    },

    "SAS": {
        "Dylan Harper": {
            "minutes": 28,
            "gp": 65
        },
        "Victor Wembanyama": {
            "minutes": 32,
            "gp": 65
        },
        "De’Aaron Fox": {
            "minutes": 26
        }
    },

    "TOR": {
        "Collin Murray-Boyles": {
            "minutes": 28,
            "gp": 75
        },
        "Allen Graves": {
            "minutes": 17,
            "gp": 55
        },
        "Jakob Poeltl": {
            "gp":70
        },
        "Kawhi Leonard": {
            "gp": 72
        },
        "RJ Barrett": {
            "gp": 66
        }
    },

    "UTA": {
        "Lauri Markkanen": {
            "gp": 65
        },
        "Keyonte George": {
            "gp": 65
        },
        "Darryn Peterson": {
            "minutes": 28,
            "gp": 65
        },
        "Jaren Jackson Jr.": {
            "minutes": 31
        },
        "Hayden Gray": {
            "gp": 40
        },
        "Josh Okogie": {
            "minutes": 25
        },
        "Mo Bamba": {
            "minutes": 20,
            "gp": 50
        }
    },

    "WAS": {
        "AJ Dybantsa": {
            "gp": 50
        },
        "Trae Young": {
            "minutes": 28,
            "gp": 45

        },
        "Anthony Davis": {
            "minutes": 28,
            "gp": 45
        },
        "Bub Carrington": {
            "gp": 50
        }
    }
}

In [47]:
team_rosters = apply_manual_rotation_fixes(
    team_rosters,
    season="26_27",
    fixes=manual_fixes
)

In [48]:
# Helper function that normalizes minutes and stats for a 9-man rotation of a team to 240 minutes.
# Rules:
#     - Convert Min to numeric values.
#     - Calculate the current total minutes for the 9-man rotation.
#     - Scale each player's minutes proportionally so the total equals 240.
#     - Preserve the relative distribution of minutes between players.
# Returns: The normalized rotation as a copy of the original DataFrame.
def normalize_rotation(rotation):
    rotation = rotation.copy()

    rotation["Min"] = pd.to_numeric(
        rotation["Min"],
        errors="coerce"
    ).fillna(0)

    # Preserve original minutes
    rotation["OriginalMin"] = rotation["Min"]

    total_minutes = rotation["Min"].sum()

    if total_minutes <= 0:
        return rotation

    # Scale minutes
    rotation["Min"] = (
        rotation["Min"] / total_minutes
    ) * 240

    # Scale per-game stats to the new minutes
    stat_columns = [
        'Ast',
        'DReb',
        '3PM',
        'FTM',
        'PTSOFFTO',
        'Pts_Paint', 'Pts_MidRange', 'Pts_3PT'
    ]
    for stat in stat_columns:
        rotation[stat] = (
            rotation[stat] *
            rotation["Min"] /
            rotation["OriginalMin"]
        )

    return rotation

In [49]:
# Helper fucntion that subtracts 1 from GP for every player in the 9-man rotation.
# team_df: Mutable team roster.
# rotation:Current 9-man rotation.
def subtract_games_Played(team_df, rotation):
    player_ids = rotation["PlayerID"]
    mask = team_df["PlayerID"].isin(player_ids)
    team_df.loc[mask, "RemainingGames"] -= 1
    return team_df

Debugging (Prints the total roster metrics and rotation information)

In [50]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

columns_to_display = [
    "PlayerName",
    "Pos",
    "TeamAbbv",
    "GP",
    "Min",
    "Ast",
    "BLK",
    "DReb",
    "FTM",
    "3PM",
    "PTSOFFTO",
    "Pts_Paint"
]

season = "26_27"

for team in sorted(team_rosters[season].keys()):

    # Prints the full roster df
    print(f"{team} Roster")

    roster = team_rosters[season][team].copy()

    roster["Min"] = pd.to_numeric(
        roster["Min"],
        errors="coerce"
    )

    display(
        roster
        .sort_values("Min", ascending=False)[
            columns_to_display
        ]
    )

    # Get 9 man rotation
    rotation = get_9_man_rotation(
        team_rosters[season][team]
    )

    rotation["Min"] = pd.to_numeric(
        rotation["Min"],
        errors="coerce"
    ).fillna(0)

    # 9 man rotation df
    print(f"\n{team} 9-man rotation")

    display(
        rotation
        .sort_values("Min", ascending=False)[
            columns_to_display
        ]
    )

    # Primary positions
    rotation["PrimaryPos"] = (
        rotation["Pos"]
        .fillna("")
        .astype(str)
        .str.upper()
        .str.split("-")
        .str[0]
        .str.strip()
    )
    
    # Split by primary positions
    guards = rotation[
        rotation["PrimaryPos"] == "G"
    ].sort_values(
        "Min",
        ascending=False
    )

    forwards = rotation[
        rotation["PrimaryPos"] == "F"
    ].sort_values(
        "Min",
        ascending=False
    )

    centers = rotation[
        rotation["PrimaryPos"] == "C"
    ].sort_values(
        "Min",
        ascending=False
    )

    # Minute totals
    guard_minutes = guards["Min"].sum()
    forward_minutes = forwards["Min"].sum()
    center_minutes = centers["Min"].sum()

    total_minutes = rotation["Min"].sum()

    position_total = (
        guard_minutes
        + forward_minutes
        + center_minutes
    )

    unassigned_minutes = (
        total_minutes
        - position_total
    )

    # Exact rotation print format
    print(f"\n{team} 9 man rotation")

    # Guards
    print("\nGuards")

    for _, player in guards.iterrows():

        print(
            f"{player['PlayerName']:<25} "
            f"{player['Min']:>5.1f} min  "
            f"({player['Pos']})"
        )

    print(
        f"Guard total: "
        f"{guard_minutes:.1f} minutes"
    )

    # Forwards
    print("\nForwards")

    for _, player in forwards.iterrows():

        print(
            f"{player['PlayerName']:<25} "
            f"{player['Min']:>5.1f} min  "
            f"({player['Pos']})"
        )

    print(
        f"Forward total: "
        f"{forward_minutes:.1f} minutes"
    )

    # Centers
    print("\nCenters")

    for _, player in centers.iterrows():

        print(
            f"{player['PlayerName']:<25} "
            f"{player['Min']:>5.1f} min  "
            f"({player['Pos']})"
        )

    print(
        f"Center total: "
        f"{center_minutes:.1f} minutes"
    )

    # Totals
    print(
        f"Total rotation minutes: {total_minutes:.1f}"
    )

    print(
        f"Position total:         {position_total:.1f}"
    )

    print(
        f"Unassigned minutes:     {unassigned_minutes:.1f}"
    )

ATL Roster


,PlayerName,Pos,TeamAbbv,GP,Min,Ast,BLK,DReb,FTM,3PM,PTSOFFTO,Pts_Paint
231,Jalen Johnson,F,ATL,72.000000,35.161088,7.861111,0.430556,8.875000,4.180556,1.666667,4.100000,11.611111
422,Nickeil Alexander-Walker,G,ATL,78.000000,33.374060,3.666667,0.538462,2.769231,3.551282,3.217949,3.700000,6.435897
442,Onyeka Okongwu,F-C,ATL,74.000000,31.040113,3.121622,1.067568,5.702703,2.067568,1.945946,2.100000,7.189189
69,CJ McCollum,G,ATL,76.000000,29.781206,3.855263,0.460526,2.500000,2.407895,2.513158,2.700000,6.210526
167,Dyson Daniels,G,ATL,76.000000,29.000000,5.168007,0.356811,3.867373,0.863253,0.253221,1.749526,8.586488
367,Luguentz Dort,G,ATL,69.000000,28.000000,1.287362,0.408927,2.862488,0.666399,1.938616,1.463049,1.847743
273,Jock Landale,C,ATL,68.000000,22.148186,1.691176,0.529412,3.000000,1.176471,1.088235,1.400000,5.882353
6,Aaron Wiggins,G,ATL,65.000000,21.753538,1.676923,0.430769,2.430769,0.815385,1.476923,2.000000,3.876923
484,Ryan Nembhard,G,ATL,60.000000,19.514667,5.266667,0.033333,2.000000,0.416667,0.800000,1.000000,2.400000
621,Kingston Flemings,G,ATL,2.657178,19.406816,3.832737,0.109483,1.340224,0.816844,1.290178,1.010489,1.975077


Available players: 22

ATL 9-man rotation


,PlayerName,Pos,TeamAbbv,GP,Min,Ast,BLK,DReb,FTM,3PM,PTSOFFTO,Pts_Paint
0,Jalen Johnson,F,ATL,72.0,35.161088,7.861111,0.430556,8.875000,4.180556,1.666667,4.100000,11.611111
1,Nickeil Alexander-Walker,G,ATL,78.0,33.374060,3.666667,0.538462,2.769231,3.551282,3.217949,3.700000,6.435897
2,Onyeka Okongwu,F-C,ATL,74.0,31.040113,3.121622,1.067568,5.702703,2.067568,1.945946,2.100000,7.189189
3,CJ McCollum,G,ATL,76.0,29.781206,3.855263,0.460526,2.500000,2.407895,2.513158,2.700000,6.210526
4,Dyson Daniels,G,ATL,76.0,29.000000,5.168007,0.356811,3.867373,0.863253,0.253221,1.749526,8.586488
5,Luguentz Dort,G,ATL,69.0,28.000000,1.287362,0.408927,2.862488,0.666399,1.938616,1.463049,1.847743
6,Jock Landale,C,ATL,68.0,22.148186,1.691176,0.529412,3.000000,1.176471,1.088235,1.400000,5.882353
7,Aaron Wiggins,G,ATL,65.0,21.753538,1.676923,0.430769,2.430769,0.815385,1.476923,2.000000,3.876923
8,Ryan Nembhard,G,ATL,60.0,19.514667,5.266667,0.033333,2.000000,0.416667,0.800000,1.000000,2.400000



ATL 9 man rotation

Guards
Nickeil Alexander-Walker   33.4 min  (G)
CJ McCollum                29.8 min  (G)
Dyson Daniels              29.0 min  (G)
Luguentz Dort              28.0 min  (G)
Aaron Wiggins              21.8 min  (G)
Ryan Nembhard              19.5 min  (G)
Guard total: 161.4 minutes

Forwards
Jalen Johnson              35.2 min  (F)
Onyeka Okongwu             31.0 min  (F-C)
Forward total: 66.2 minutes

Centers
Jock Landale               22.1 min  (C)
Center total: 22.1 minutes
Total rotation minutes: 249.8
Position total:         249.8
Unassigned minutes:     0.0
BKN Roster


...


UTA 9 man rotation

Guards
Keyonte George             33.1 min  (G)
Darryn Peterson            28.0 min  (G)
Isaiah Collier             25.7 min  (G)
Josh Okogie                25.0 min  (G)
Hayden Gray                24.8 min  (G)
Guard total: 136.5 minutes

Forwards
Lauri Markkanen            34.3 min  (F-C)
Jaren Jackson Jr.          31.0 min  (F-C)
Ace Bailey                 27.6 min  (F)
Forward total: 93.0 minutes

Centers
Jusuf Nurkić               26.4 min  (C)
Center total: 26.4 minutes
Total rotation minutes: 255.9
Position total:         255.9
Unassigned minutes:     0.0
WAS Roster


,PlayerName,Pos,TeamAbbv,GP,Min,Ast,BLK,DReb,FTM,3PM,PTSOFFTO,Pts_Paint
352,Kyshawn George,F,WAS,48.000000,28.978403,4.458333,0.895833,4.208333,2.187500,2.062500,1.600000,4.750000
688,Trae Young,G,WAS,45.000000,28.000000,8.995787,0.122670,2.044497,5.019240,2.228502,2.952254,0.838244
677,Anthony Davis,F-C,WAS,45.000000,28.000000,2.971583,1.805934,7.486419,4.695429,0.574615,2.930539,4.301407
65,Bub Carrington,G,WAS,50.000000,27.690854,4.646341,0.207317,2.963415,0.987805,2.060976,1.500000,1.609756
136,Deandre Ayton,C,WAS,72.000000,27.199792,0.819444,0.972222,5.416667,1.263889,0.000000,2.100000,10.777778
16,Alex Sarr,C,WAS,48.000000,27.180451,2.708333,1.979167,5.208333,2.104167,1.041667,2.000000,10.125000
44,Bilal Coulibaly,G,WAS,56.000000,26.207232,2.589286,0.964286,3.482143,2.357143,1.160714,2.100000,5.428571
652,AJ Dybantsa,F,WAS,50.000000,24.540266,2.256040,0.208463,3.197713,2.078903,0.774349,2.043655,6.217963
528,Tre Johnson,G,WAS,60.000000,24.046944,2.033333,0.300000,2.350000,1.266667,1.916667,1.700000,3.366667
337,Khris Middleton,F,WAS,63.000000,22.842646,2.793651,0.142857,2.857143,1.888889,1.063492,1.000000,2.349206


Available players: 18

WAS 9-man rotation


,PlayerName,Pos,TeamAbbv,GP,Min,Ast,BLK,DReb,FTM,3PM,PTSOFFTO,Pts_Paint
0,Kyshawn George,F,WAS,48.0,28.978403,4.458333,0.895833,4.208333,2.187500,2.062500,1.600000,4.750000
1,Trae Young,G,WAS,45.0,28.000000,8.995787,0.122670,2.044497,5.019240,2.228502,2.952254,0.838244
2,Anthony Davis,F-C,WAS,45.0,28.000000,2.971583,1.805934,7.486419,4.695429,0.574615,2.930539,4.301407
3,Bub Carrington,G,WAS,50.0,27.690854,4.646341,0.207317,2.963415,0.987805,2.060976,1.500000,1.609756
4,Deandre Ayton,C,WAS,72.0,27.199792,0.819444,0.972222,5.416667,1.263889,0.000000,2.100000,10.777778
5,Alex Sarr,C,WAS,48.0,27.180451,2.708333,1.979167,5.208333,2.104167,1.041667,2.000000,10.125000
6,Bilal Coulibaly,G,WAS,56.0,26.207232,2.589286,0.964286,3.482143,2.357143,1.160714,2.100000,5.428571
7,AJ Dybantsa,F,WAS,50.0,24.540266,2.256040,0.208463,3.197713,2.078903,0.774349,2.043655,6.217963
8,Tre Johnson,G,WAS,60.0,24.046944,2.033333,0.300000,2.350000,1.266667,1.916667,1.700000,3.366667



WAS 9 man rotation

Guards
Trae Young                 28.0 min  (G)
Bub Carrington             27.7 min  (G)
Bilal Coulibaly            26.2 min  (G)
Tre Johnson                24.0 min  (G)
Guard total: 105.9 minutes

Forwards
Kyshawn George             29.0 min  (F)
Anthony Davis              28.0 min  (F-C)
AJ Dybantsa                24.5 min  (F)
Forward total: 81.5 minutes

Centers
Deandre Ayton              27.2 min  (C)
Alex Sarr                  27.2 min  (C)
Center total: 54.4 minutes
Total rotation minutes: 241.8
Position total:         241.8
Unassigned minutes:     0.0


# Step 11: Create Conferences/Standings

In [51]:
# Helper function that updates W/L records and winning percentage for a completed game.
def update_standings(standings, winner, loser):

    def find_conference(team):
        if team in standings["East"]:
            return "East"
        elif team in standings["West"]:
            return "West"
        else:
            raise ValueError(f"Unknown team abbreviation: {team}")

    winner_conf = find_conference(winner)
    loser_conf = find_conference(loser)

    # Update records
    standings[winner_conf][winner]["W"] += 1
    standings[loser_conf][loser]["L"] += 1

    # Recalculate winning percentages for any conference(s) touched
    for conf in {winner_conf, loser_conf}:
        for team, record in standings[conf].items():
            games = record["W"] + record["L"]
            record["W_Pct"] = record["W"] / games if games > 0 else 0.0

    return standings

In [52]:
# Helper function that creates initial NBA standings using TeamAbbv.
# Returns:
#     standings = {
#         "East": {...},
#         "West": {...}
#     }

# Each team starts with:
#     W = 0
#     L = 0
#     W_Pct = 0
def create_standings():

    eastern_teams = [
        "ATL",  # Atlanta Hawks
        "BOS",  # Boston Celtics
        "BKN",  # Brooklyn Nets
        "CHA",  # Charlotte Hornets
        "CHI",  # Chicago Bulls
        "CLE",  # Cleveland Cavaliers
        "DET",  # Detroit Pistons
        "IND",  # Indiana Pacers
        "MIA",  # Miami Heat
        "MIL",  # Milwaukee Bucks
        "NYK",  # New York Knicks
        "ORL",  # Orlando Magic
        "PHI",  # Philadelphia 76ers
        "TOR",  # Toronto Raptors
        "WAS",  # Washington Wizards
    ]

    western_teams = [
        "DAL",  # Dallas Mavericks
        "DEN",  # Denver Nuggets
        "GSW",  # Golden State Warriors
        "HOU",  # Houston Rockets
        "LAC",  # LA Clippers
        "LAL",  # Los Angeles Lakers
        "MEM",  # Memphis Grizzlies
        "MIN",  # Minnesota Timberwolves
        "NOP",  # New Orleans Pelicans
        "OKC",  # Oklahoma City Thunder
        "PHX",  # Phoenix Suns
        "POR",  # Portland Trail Blazers
        "SAC",  # Sacramento Kings
        "SAS",  # San Antonio Spurs
        "UTA",  # Utah Jazz
    ]

    standings = {
        "East": {},
        "West": {}
    }

    for team in eastern_teams:
        standings["East"][team] = {
            "W": 0,
            "L": 0,
            "W_Pct": 0.0
        }

    for team in western_teams:
        standings["West"][team] = {
            "W": 0,
            "L": 0,
            "W_Pct": 0.0
        }

    return standings


In [53]:
# Helper function that prints current NBA standings for both conferences.
def print_standings(standings):

    print("Eastern Conference")

    east_sorted = sorted(
        standings["East"].items(),
        key=lambda x: x[1]["W_Pct"],
        reverse=True
    )

    print(
        f"{'Team':<6}"
        f"{'W':>5}"
        f"{'L':>5}"
        f"{'W_Pct':>10}"
    )

    for team, record in east_sorted:
        print(
            f"{team:<6}"
            f"{record['W']:>5}"
            f"{record['L']:>5}"
            f"{record['W_Pct']:>10.3f}"
        )

    print("Western Conference")

    west_sorted = sorted(
        standings["West"].items(),
        key=lambda x: x[1]["W_Pct"],
        reverse=True
    )

    print(
        f"{'Team':<6}"
        f"{'W':>5}"
        f"{'L':>5}"
        f"{'W_Pct':>10}"
    )

    for team, record in west_sorted:
        print(
            f"{team:<6}"
            f"{record['W']:>5}"
            f"{record['L']:>5}"
            f"{record['W_Pct']:>10.3f}"
        )

In [54]:
# Helper function that compares the standing that the model obtained with the actual standings.
def compare_standings(model_standings, actual_standings):

    comparison = {
        "East": {},
        "West": {}
    }

    for conference in ["East", "West"]:

        for team, model_record in model_standings[conference].items():

            actual_record = actual_standings[conference][team]

            model_w = model_record["W"]
            model_l = model_record["L"]

            actual_w = actual_record["W"]
            actual_l = actual_record["L"]

            model_pct = model_record["W_Pct"]

            actual_games = actual_w + actual_l
            actual_pct = (
                actual_w / actual_games
                if actual_games > 0
                else 0.0
            )

            comparison[conference][team] = {
                "Model_W": model_w,
                "Actual_W": actual_w,
                "W_Diff": model_w - actual_w,

                "Model_L": model_l,
                "Actual_L": actual_l,
                "L_Diff": model_l - actual_l,

                "Model_W_Pct": model_pct,
                "Actual_W_Pct": actual_pct,

                "W_Pct_Diff": model_pct - actual_pct
            }

    return comparison

In [55]:
# Helper function that takes into account home-court advantage.
#     The boost is a logit-space shift; 0.20 corresponds to roughly a 50/50 game
#     becoming ~55/45 in the home team's favor (the NBA's long-run home edge).
def apply_home_advantage(home_prob, away_prob, boost=0.20):
    def logit(p):
        p = np.clip(p, 1e-6, 1 - 1e-6)
        return np.log(p / (1 - p))

    def sigmoid(x):
        return 1 / (1 + np.exp(-x))

    home_prob_adj = sigmoid(logit(home_prob) + boost)
    away_prob_adj = sigmoid(logit(away_prob) - boost)

    return home_prob_adj, away_prob_adj

Debugging

# Step 12: Get schedule

In [56]:
# Picking the season
SEASONS = [
    # "21_22",
    # "22_23",
    # "23_24",
    # "24_25",
    # "25_26",
    "26_27",
]

Load Model

In [57]:
model = xgb.Booster()
model.load_model("/Users/ronanhwang/Downloads/Github/nba_2020s/best_model_Altogether.json")

Run model of schedule

In [58]:
# Helper function that applies a penalty to a team that is playing a back to back
def apply_back_to_back_penalty(home_prediction, away_prediction, home_is_b2b, away_is_b2b, penalty=0.03):
    if home_is_b2b and not away_is_b2b:
        home_prediction -= penalty
        away_prediction += penalty
    elif away_is_b2b and not home_is_b2b:
        away_prediction -= penalty
        home_prediction += penalty
    # if both teams are on a b2b, no relative adjustment
    return home_prediction, away_prediction

In [59]:
SEASON_TIPOFF = {
    "21_22": "2021-10-19",
    "22_23": "2022-10-18",
    "23_24": "2023-10-24",
    "24_25": "2024-10-22",
    "25_26": "2025-10-21",
    "26_27": "2026-10-20",
}

def process_all_seasons(data, team_rosters, seasons):

    results = {}

    for season in seasons:

        print(f"\nProcessing season: {season}")

        schedule = data[season]["ScheduleLeagueV2"].copy()

        schedule["gameDateTimeEst"] = pd.to_datetime(
            schedule["gameDateTimeEst"],
            utc=True
        )

        tipoff = pd.Timestamp(
            SEASON_TIPOFF[season],
            tz="UTC"
        )

        season_rosters = {
            team: df.copy()
            for team, df in team_rosters[season].items()
        }

        standings = create_standings()

        schedule = (
            schedule
            .sort_values("gameDateTimeEst")
            .reset_index(drop=True)
        )

        correct_predictions = 0
        incorrect_predictions = 0
        flipped_by_home_boost = 0
        flipped_by_b2b = 0

        processed_games = 0
        skipped_non_regular = 0
        skipped_not_final = 0

        team_games_played = {
            team: 0
            for team in season_rosters
        }

        # Store predictions for future seasons
        season_predictions = []

        # Last game date per team
        last_game_date = {
            team: None
            for team in season_rosters
        }

        for _, game in schedule.iterrows():

            # Game Type
            label = (
                str(game["gameLabel"]).strip()
                if pd.notna(game["gameLabel"])
                else ""
            )

            series = (
                str(game["seriesText"]).strip()
                if pd.notna(game["seriesText"])
                else ""
            )

            # Skip non regular season games
            # Game Type
            label = (
                str(game["gameLabel"]).strip()
                if pd.notna(game["gameLabel"])
                else ""
            )

            series = (
                str(game["seriesText"]).strip()
                if pd.notna(game["seriesText"])
                else ""
            )

            # Skip ONLY:
            # 1. Preseason
            # 2. Emirates NBA Cup games at Neutral Site
            is_non_regular = (
                label == "Preseason"
                or (
                    label == "Emirates NBA Cup"
                    and pd.isna(game["homeTeam_teamTricode"])
                    and pd.isna(game["awayTeam_teamTricode"])
                )
            )
            if is_non_regular:
                skipped_non_regular += 1
                continue

            # Process scheduled games 
            status = (
                str(game["gameStatusText"]).strip()
                if pd.notna(game["gameStatusText"])
                else ""
            )

            is_final = status.lower().startswith("final")

            if season != "26_27" and not is_final:
                skipped_not_final += 1
                continue

            # Teams
            home_team = game["homeTeam_teamTricode"]
            away_team = game["awayTeam_teamTricode"]

            if (
                home_team not in season_rosters
                or away_team not in season_rosters
            ):
                print(
                    f"Skipping game with unknown team(s): "
                    f"{away_team} @ {home_team}"
                )
                continue

            processed_games += 1

            # Eastern Calendar Date for B2B games
            game_date = (
                game["gameDateTimeEst"]
                .tz_convert("America/New_York")
                .normalize()
            )

            home_is_b2b = (
                last_game_date[home_team] is not None
                and (
                    game_date - last_game_date[home_team]
                ).days == 1
            )

            away_is_b2b = (
                last_game_date[away_team] is not None
                and (
                    game_date - last_game_date[away_team]
                ).days == 1
            )

            # Current Rosters
            home_roster = season_rosters[home_team]
            away_roster = season_rosters[away_team]

            # 9-MAN Rotations
            home_rotation = get_9_man_rotation(home_roster)
            away_rotation = get_9_man_rotation(away_roster)

            home_rotation = normalize_rotation(home_rotation)
            away_rotation = normalize_rotation(away_rotation)

            # Aggregate team stats
            homeTeam = home_rotation.agg({
                "Ast": "sum",
                "BLK": "sum",
                "DReb": "sum",
                "FTM": "sum",
                "3PM": "sum",
                "PTSOFFTO": "sum",
                "Pts_Paint": "sum"
            })

            awayTeam = away_rotation.agg({
                "Ast": "sum",
                "BLK": "sum",
                "DReb": "sum",
                "FTM": "sum",
                "3PM": "sum",
                "PTSOFFTO": "sum",
                "Pts_Paint": "sum"
            })

            # Feature Differences
            diff_homeTeam = pd.Series({
                "diff_Ast":
                    homeTeam["Ast"] - awayTeam["Ast"],

                "diff_OppPointsInPaint":
                    homeTeam["Pts_Paint"] - awayTeam["Pts_Paint"],

                "diff_PointsOffTO":
                    homeTeam["PTSOFFTO"] - awayTeam["PTSOFFTO"],

                "diff_Blk":
                    homeTeam["BLK"] - awayTeam["BLK"],

                "diff_DReb":
                    homeTeam["DReb"] - awayTeam["DReb"],

                "diff_FTM":
                    homeTeam["FTM"] - awayTeam["FTM"],

                "diff_3PM":
                    homeTeam["3PM"] - awayTeam["3PM"]
            })

            diff_awayTeam = pd.Series({
                "diff_Ast":
                    awayTeam["Ast"] - homeTeam["Ast"],

                "diff_OppPointsInPaint":
                    awayTeam["Pts_Paint"] - homeTeam["Pts_Paint"],

                "diff_PointsOffTO":
                    awayTeam["PTSOFFTO"] - homeTeam["PTSOFFTO"],

                "diff_Blk":
                    awayTeam["BLK"] - homeTeam["BLK"],

                "diff_DReb":
                    awayTeam["DReb"] - homeTeam["DReb"],

                "diff_FTM":
                    awayTeam["FTM"] - homeTeam["FTM"],

                "diff_3PM":
                    awayTeam["3PM"] - homeTeam["3PM"]
            })

            features = [
                "diff_Ast",
                "diff_OppPointsInPaint",
                "diff_PointsOffTO",
                "diff_Blk",
                "diff_DReb",
                "diff_FTM",
                "diff_3PM"
            ]

            home_team_stats = (
                diff_homeTeam
                .to_frame()
                .T[features]
            )

            away_team_stats = (
                diff_awayTeam
                .to_frame()
                .T[features]
            )

            # XGBoost Prediction
            home_dmatrix = xgb.DMatrix(
                home_team_stats,
                feature_names=features
            )

            away_dmatrix = xgb.DMatrix(
                away_team_stats,
                feature_names=features
            )

            home_prediction = model.predict(
                home_dmatrix
            )[0]

            away_prediction = model.predict(
                away_dmatrix
            )[0]

            # Pre boost prediction
            pre_boost_home_win = home_prediction >= 0.5

            # Home court Advantage
            home_prediction, away_prediction = (
                apply_home_advantage(
                    home_prediction,
                    away_prediction
                )
            )

            post_home_boost_win = home_prediction >= 0.5

            if post_home_boost_win != pre_boost_home_win:
                flipped_by_home_boost += 1

            # Back to Back Penalty
            home_prediction, away_prediction = (
                apply_back_to_back_penalty(
                    home_prediction,
                    away_prediction,
                    home_is_b2b,
                    away_is_b2b
                )
            )

            home_win = home_prediction >= 0.5
            away_win = away_prediction >= 0.5

            if home_win != post_home_boost_win:
                flipped_by_b2b += 1

            # Determine Model Winner
            if home_win == away_win:

                if home_prediction > away_prediction:
                    model_winner = home_team
                    model_loser = away_team

                elif away_prediction > home_prediction:
                    model_winner = away_team
                    model_loser = home_team

                else:
                    model_winner = home_team
                    model_loser = away_team

            else:

                if home_win:
                    model_winner = home_team
                    model_loser = away_team

                else:
                    model_winner = away_team
                    model_loser = home_team

            # Save 26_27 Prediction
            if season == "26_27":

                season_predictions.append({
                    "gameId": game["gameId"],
                    "gameDate": game["gameDate"],
                    "gameDateTimeEst": game["gameDateTimeEst"],
                    "away_team": away_team,
                    "home_team": home_team,
                    "predicted_winner": model_winner,
                    "predicted_loser": model_loser,
                    "away_prediction": away_prediction,
                    "home_prediction": home_prediction,
                    "away_is_b2b": away_is_b2b,
                    "home_is_b2b": home_is_b2b,
                })

            # Update Standings
            # Update Standings
            update_standings(
                standings,
                model_winner,
                model_loser
            )

            # Track actual games processed for each team
            team_games_played[home_team] += 1
            team_games_played[away_team] += 1
            # Update Rotations
            subtract_games_Played(
                home_roster,
                home_rotation
            )

            subtract_games_Played(
                away_roster,
                away_rotation
            )

            # Update last game date
            last_game_date[home_team] = game_date
            last_game_date[away_team] = game_date

        # Save Results
        results[season] = {
            "team_rosters": season_rosters,
            "standings": standings,
            "correct_predictions": correct_predictions,
            "incorrect_predictions": incorrect_predictions,
            "flipped_by_home_boost": flipped_by_home_boost,
            "flipped_by_b2b": flipped_by_b2b,
            "predictions": season_predictions
        }
        
        # Output
        print(
            f"\nFinished {season} — "
        )

        print("\nSCHEDULE DIAGNOSTICS")
        print(f"Total schedule rows:       {len(schedule)}")
        print(f"Skipped non-regular:       {skipped_non_regular}")
        print(f"Skipped not final:         {skipped_not_final}")
        print(f"Processed games:           {processed_games}")

        if season == "26_27":
            print(
                f"26_27 predictions stored:  "
                f"{len(season_predictions)}"
            )
        print(
            f"Games flipped by home-court boost: "
            f"{flipped_by_home_boost}"
        )
    return results


In [60]:
results = process_all_seasons(
    data,
    team_rosters,
    SEASONS
)


Processing season: 26_27
Available players: 20
Available players: 18
Available players: 15
  -> Adding real player Andre Drummond (C, 19.54 min) to fill C
     Removing Jose Alvarado to make room.
Available players: 19
Available players: 17
  -> Adding real player Luke Kornet (C-F, 21.03 min) to fill C
     Removing Keldon Johnson to make room.
Available players: 17
Available players: 18
Available players: 18
Available players: 18
Available players: 22
Available players: 18
Available players: 23
Available players: 19
  -> Adding real player Allen Graves (F-C, 17.00 min) to fill C
     Removing Ja'Kobe Walter to make room.
Available players: 16
Available players: 19
  -> Adding real player Nick Richards (C, 14.60 min) to fill C
     Removing Nikola Jović to make room.
Available players: 16
  -> Adding real player Rocco Zikarsky (C, 12.00 min) to fill C
     Removing Bones Hyland to make room.
Available players: 20
  -> Adding real player Yves Missi (C, 19.65 min) to fill C
     Removin

# Step 13: Check

In [61]:
# NBA standings from previous years
nba_standings = {
    "21_22": {
        "west": [
            {"teamAbbv": "PHX", "W": 64, "L": 18, "W_Pct": 0.780},
            {"teamAbbv": "MEM", "W": 56, "L": 26, "W_Pct": 0.683},
            {"teamAbbv": "GSW", "W": 53, "L": 29, "W_Pct": 0.646},
            {"teamAbbv": "DAL", "W": 52, "L": 30, "W_Pct": 0.634},
            {"teamAbbv": "UTA", "W": 49, "L": 33, "W_Pct": 0.598},
            {"teamAbbv": "DEN", "W": 48, "L": 34, "W_Pct": 0.585},
            {"teamAbbv": "MIN", "W": 46, "L": 36, "W_Pct": 0.561},
            {"teamAbbv": "LAC", "W": 42, "L": 40, "W_Pct": 0.512},
            {"teamAbbv": "NOP", "W": 36, "L": 46, "W_Pct": 0.439},
            {"teamAbbv": "SAS", "W": 34, "L": 48, "W_Pct": 0.415},
            {"teamAbbv": "LAL", "W": 33, "L": 49, "W_Pct": 0.402},
            {"teamAbbv": "SAC", "W": 30, "L": 52, "W_Pct": 0.366},
            {"teamAbbv": "POR", "W": 27, "L": 55, "W_Pct": 0.329},
            {"teamAbbv": "OKC", "W": 24, "L": 58, "W_Pct": 0.293},
            {"teamAbbv": "HOU", "W": 20, "L": 62, "W_Pct": 0.244},
        ],
        "east": [
            {"teamAbbv": "MIA", "W": 53, "L": 29, "W_Pct": 0.646},
            {"teamAbbv": "BOS", "W": 51, "L": 31, "W_Pct": 0.622},
            {"teamAbbv": "MIL", "W": 51, "L": 31, "W_Pct": 0.622},
            {"teamAbbv": "PHI", "W": 51, "L": 31, "W_Pct": 0.622},
            {"teamAbbv": "TOR", "W": 48, "L": 34, "W_Pct": 0.585},
            {"teamAbbv": "CHI", "W": 46, "L": 36, "W_Pct": 0.561},
            {"teamAbbv": "BKN", "W": 44, "L": 38, "W_Pct": 0.537},
            {"teamAbbv": "CLE", "W": 44, "L": 38, "W_Pct": 0.537},
            {"teamAbbv": "ATL", "W": 43, "L": 39, "W_Pct": 0.524},
            {"teamAbbv": "CHA", "W": 43, "L": 39, "W_Pct": 0.524},
            {"teamAbbv": "NYK", "W": 37, "L": 45, "W_Pct": 0.451},
            {"teamAbbv": "WAS", "W": 35, "L": 47, "W_Pct": 0.427},
            {"teamAbbv": "IND", "W": 25, "L": 57, "W_Pct": 0.305},
            {"teamAbbv": "DET", "W": 23, "L": 59, "W_Pct": 0.280},
            {"teamAbbv": "ORL", "W": 22, "L": 60, "W_Pct": 0.268},
        ],
    },

    "22_23": {
        "east": [
            {"teamAbbv": "MIL", "W": 58, "L": 24, "W_Pct": 0.707},
            {"teamAbbv": "BOS", "W": 57, "L": 25, "W_Pct": 0.695},
            {"teamAbbv": "PHI", "W": 54, "L": 28, "W_Pct": 0.659},
            {"teamAbbv": "CLE", "W": 51, "L": 31, "W_Pct": 0.622},
            {"teamAbbv": "NYK", "W": 47, "L": 35, "W_Pct": 0.573},
            {"teamAbbv": "BKN", "W": 45, "L": 37, "W_Pct": 0.549},
            {"teamAbbv": "MIA", "W": 44, "L": 38, "W_Pct": 0.537},
            {"teamAbbv": "ATL", "W": 41, "L": 41, "W_Pct": 0.500},
            {"teamAbbv": "TOR", "W": 41, "L": 41, "W_Pct": 0.500},
            {"teamAbbv": "CHI", "W": 40, "L": 42, "W_Pct": 0.488},
            {"teamAbbv": "IND", "W": 35, "L": 47, "W_Pct": 0.427},
            {"teamAbbv": "WAS", "W": 35, "L": 47, "W_Pct": 0.427},
            {"teamAbbv": "ORL", "W": 34, "L": 48, "W_Pct": 0.415},
            {"teamAbbv": "CHA", "W": 27, "L": 55, "W_Pct": 0.329},
            {"teamAbbv": "DET", "W": 17, "L": 65, "W_Pct": 0.207},
        ],
        "west": [
            {"teamAbbv": "DEN", "W": 53, "L": 29, "W_Pct": 0.646},
            {"teamAbbv": "MEM", "W": 51, "L": 31, "W_Pct": 0.622},
            {"teamAbbv": "SAC", "W": 48, "L": 34, "W_Pct": 0.585},
            {"teamAbbv": "PHX", "W": 45, "L": 37, "W_Pct": 0.549},
            {"teamAbbv": "LAC", "W": 44, "L": 38, "W_Pct": 0.537},
            {"teamAbbv": "GSW", "W": 44, "L": 38, "W_Pct": 0.537},
            {"teamAbbv": "LAL", "W": 43, "L": 39, "W_Pct": 0.524},
            {"teamAbbv": "MIN", "W": 42, "L": 40, "W_Pct": 0.512},
            {"teamAbbv": "NOP", "W": 42, "L": 40, "W_Pct": 0.512},
            {"teamAbbv": "OKC", "W": 40, "L": 42, "W_Pct": 0.488},
            {"teamAbbv": "DAL", "W": 38, "L": 44, "W_Pct": 0.463},
            {"teamAbbv": "UTA", "W": 37, "L": 45, "W_Pct": 0.451},
            {"teamAbbv": "POR", "W": 33, "L": 49, "W_Pct": 0.402},
            {"teamAbbv": "HOU", "W": 22, "L": 60, "W_Pct": 0.268},
            {"teamAbbv": "SAS", "W": 22, "L": 60, "W_Pct": 0.268},
        ],
    },

    "23_24": {
        "east": [
            {"teamAbbv": "BOS", "W": 64, "L": 18, "W_Pct": 0.780},
            {"teamAbbv": "NYK", "W": 50, "L": 32, "W_Pct": 0.610},
            {"teamAbbv": "MIL", "W": 49, "L": 33, "W_Pct": 0.598},
            {"teamAbbv": "CLE", "W": 48, "L": 34, "W_Pct": 0.585},
            {"teamAbbv": "ORL", "W": 47, "L": 35, "W_Pct": 0.573},
            {"teamAbbv": "IND", "W": 47, "L": 35, "W_Pct": 0.573},
            {"teamAbbv": "PHI", "W": 47, "L": 35, "W_Pct": 0.573},
            {"teamAbbv": "MIA", "W": 46, "L": 36, "W_Pct": 0.561},
            {"teamAbbv": "CHI", "W": 39, "L": 43, "W_Pct": 0.476},
            {"teamAbbv": "ATL", "W": 36, "L": 46, "W_Pct": 0.439},
            {"teamAbbv": "BKN", "W": 32, "L": 50, "W_Pct": 0.390},
            {"teamAbbv": "TOR", "W": 25, "L": 57, "W_Pct": 0.305},
            {"teamAbbv": "CHA", "W": 21, "L": 61, "W_Pct": 0.256},
            {"teamAbbv": "WAS", "W": 15, "L": 67, "W_Pct": 0.183},
            {"teamAbbv": "DET", "W": 14, "L": 68, "W_Pct": 0.171},
        ],
        "west": [
            {"teamAbbv": "OKC", "W": 57, "L": 25, "W_Pct": 0.695},
            {"teamAbbv": "DEN", "W": 57, "L": 25, "W_Pct": 0.695},
            {"teamAbbv": "MIN", "W": 56, "L": 26, "W_Pct": 0.683},
            {"teamAbbv": "LAC", "W": 51, "L": 31, "W_Pct": 0.622},
            {"teamAbbv": "DAL", "W": 50, "L": 32, "W_Pct": 0.610},
            {"teamAbbv": "PHX", "W": 49, "L": 33, "W_Pct": 0.598},
            {"teamAbbv": "NOP", "W": 49, "L": 33, "W_Pct": 0.598},
            {"teamAbbv": "LAL", "W": 47, "L": 35, "W_Pct": 0.573},
            {"teamAbbv": "SAC", "W": 46, "L": 36, "W_Pct": 0.561},
            {"teamAbbv": "GSW", "W": 46, "L": 36, "W_Pct": 0.561},
            {"teamAbbv": "HOU", "W": 41, "L": 41, "W_Pct": 0.500},
            {"teamAbbv": "UTA", "W": 31, "L": 51, "W_Pct": 0.378},
            {"teamAbbv": "MEM", "W": 27, "L": 55, "W_Pct": 0.329},
            {"teamAbbv": "SAS", "W": 22, "L": 60, "W_Pct": 0.268},
            {"teamAbbv": "POR", "W": 21, "L": 61, "W_Pct": 0.256},
        ],
    },

    "24_25": {
        "east": [
            {"teamAbbv": "CLE", "W": 64, "L": 18, "W_Pct": 0.780},
            {"teamAbbv": "BOS", "W": 61, "L": 21, "W_Pct": 0.744},
            {"teamAbbv": "NYK", "W": 51, "L": 31, "W_Pct": 0.622},
            {"teamAbbv": "IND", "W": 50, "L": 32, "W_Pct": 0.610},
            {"teamAbbv": "MIL", "W": 48, "L": 34, "W_Pct": 0.585},
            {"teamAbbv": "DET", "W": 44, "L": 38, "W_Pct": 0.537},
            {"teamAbbv": "ORL", "W": 41, "L": 41, "W_Pct": 0.500},
            {"teamAbbv": "ATL", "W": 40, "L": 42, "W_Pct": 0.488},
            {"teamAbbv": "CHI", "W": 39, "L": 43, "W_Pct": 0.476},
            {"teamAbbv": "MIA", "W": 37, "L": 45, "W_Pct": 0.451},
            {"teamAbbv": "TOR", "W": 30, "L": 52, "W_Pct": 0.366},
            {"teamAbbv": "BKN", "W": 26, "L": 56, "W_Pct": 0.317},
            {"teamAbbv": "PHI", "W": 24, "L": 58, "W_Pct": 0.293},
            {"teamAbbv": "CHA", "W": 19, "L": 63, "W_Pct": 0.232},
            {"teamAbbv": "WAS", "W": 18, "L": 64, "W_Pct": 0.220},
        ],
        "west": [
            {"teamAbbv": "OKC", "W": 68, "L": 14, "W_Pct": 0.829},
            {"teamAbbv": "HOU", "W": 52, "L": 30, "W_Pct": 0.634},
            {"teamAbbv": "LAL", "W": 50, "L": 32, "W_Pct": 0.610},
            {"teamAbbv": "DEN", "W": 50, "L": 32, "W_Pct": 0.610},
            {"teamAbbv": "LAC", "W": 50, "L": 32, "W_Pct": 0.610},
            {"teamAbbv": "MIN", "W": 49, "L": 33, "W_Pct": 0.598},
            {"teamAbbv": "GSW", "W": 48, "L": 34, "W_Pct": 0.585},
            {"teamAbbv": "MEM", "W": 48, "L": 34, "W_Pct": 0.585},
            {"teamAbbv": "SAC", "W": 40, "L": 42, "W_Pct": 0.488},
            {"teamAbbv": "DAL", "W": 39, "L": 43, "W_Pct": 0.476},
            {"teamAbbv": "PHX", "W": 36, "L": 46, "W_Pct": 0.439},
            {"teamAbbv": "POR", "W": 36, "L": 46, "W_Pct": 0.439},
            {"teamAbbv": "SAS", "W": 34, "L": 48, "W_Pct": 0.415},
            {"teamAbbv": "NOP", "W": 21, "L": 61, "W_Pct": 0.256},
            {"teamAbbv": "UTA", "W": 17, "L": 65, "W_Pct": 0.207},
        ],
    },

    "25_26": {
        "east": [
            {"teamAbbv": "DET", "W": 60, "L": 22, "W_Pct": 0.732},
            {"teamAbbv": "BOS", "W": 56, "L": 26, "W_Pct": 0.683},
            {"teamAbbv": "NYK", "W": 53, "L": 29, "W_Pct": 0.646},
            {"teamAbbv": "CLE", "W": 52, "L": 30, "W_Pct": 0.634},
            {"teamAbbv": "TOR", "W": 46, "L": 36, "W_Pct": 0.561},
            {"teamAbbv": "ATL", "W": 46, "L": 36, "W_Pct": 0.561},
            {"teamAbbv": "PHI", "W": 45, "L": 37, "W_Pct": 0.549},
            {"teamAbbv": "ORL", "W": 45, "L": 37, "W_Pct": 0.549},
            {"teamAbbv": "CHA", "W": 44, "L": 38, "W_Pct": 0.537},
            {"teamAbbv": "MIA", "W": 43, "L": 39, "W_Pct": 0.524},
            {"teamAbbv": "MIL", "W": 32, "L": 50, "W_Pct": 0.390},
            {"teamAbbv": "CHI", "W": 31, "L": 51, "W_Pct": 0.378},
            {"teamAbbv": "BKN", "W": 20, "L": 62, "W_Pct": 0.244},
            {"teamAbbv": "IND", "W": 19, "L": 63, "W_Pct": 0.232},
            {"teamAbbv": "WAS", "W": 17, "L": 65, "W_Pct": 0.207},
        ],
        "west": [
            {"teamAbbv": "OKC", "W": 64, "L": 18, "W_Pct": 0.780},
            {"teamAbbv": "SAS", "W": 62, "L": 20, "W_Pct": 0.756},
            {"teamAbbv": "DEN", "W": 54, "L": 28, "W_Pct": 0.659},
            {"teamAbbv": "LAL", "W": 53, "L": 29, "W_Pct": 0.646},
            {"teamAbbv": "HOU", "W": 52, "L": 30, "W_Pct": 0.634},
            {"teamAbbv": "MIN", "W": 49, "L": 33, "W_Pct": 0.598},
            {"teamAbbv": "PHX", "W": 45, "L": 37, "W_Pct": 0.549},
            {"teamAbbv": "POR", "W": 42, "L": 40, "W_Pct": 0.512},
            {"teamAbbv": "LAC", "W": 42, "L": 40, "W_Pct": 0.512},
            {"teamAbbv": "GSW", "W": 37, "L": 45, "W_Pct": 0.451},
            {"teamAbbv": "NOP", "W": 26, "L": 56, "W_Pct": 0.317},
            {"teamAbbv": "DAL", "W": 26, "L": 56, "W_Pct": 0.317},
            {"teamAbbv": "MEM", "W": 25, "L": 57, "W_Pct": 0.305},
            {"teamAbbv": "SAC", "W": 22, "L": 60, "W_Pct": 0.268},
            {"teamAbbv": "UTA", "W": 22, "L": 60, "W_Pct": 0.268},
        ],
    },
    "26_27": {
        "east": [
            {"teamAbbv": "DET", "W": 60, "L": 22, "W_Pct": 0.732},
            {"teamAbbv": "BOS", "W": 56, "L": 26, "W_Pct": 0.683},
            {"teamAbbv": "NYK", "W": 53, "L": 29, "W_Pct": 0.646},
            {"teamAbbv": "CLE", "W": 52, "L": 30, "W_Pct": 0.634},
            {"teamAbbv": "TOR", "W": 46, "L": 36, "W_Pct": 0.561},
            {"teamAbbv": "ATL", "W": 46, "L": 36, "W_Pct": 0.561},
            {"teamAbbv": "PHI", "W": 45, "L": 37, "W_Pct": 0.549},
            {"teamAbbv": "ORL", "W": 45, "L": 37, "W_Pct": 0.549},
            {"teamAbbv": "CHA", "W": 44, "L": 38, "W_Pct": 0.537},
            {"teamAbbv": "MIA", "W": 43, "L": 39, "W_Pct": 0.524},
            {"teamAbbv": "MIL", "W": 32, "L": 50, "W_Pct": 0.390},
            {"teamAbbv": "CHI", "W": 31, "L": 51, "W_Pct": 0.378},
            {"teamAbbv": "BKN", "W": 20, "L": 62, "W_Pct": 0.244},
            {"teamAbbv": "IND", "W": 19, "L": 63, "W_Pct": 0.232},
            {"teamAbbv": "WAS", "W": 17, "L": 65, "W_Pct": 0.207},
        ],
        "west": [
            {"teamAbbv": "OKC", "W": 64, "L": 18, "W_Pct": 0.780},
            {"teamAbbv": "SAS", "W": 62, "L": 20, "W_Pct": 0.756},
            {"teamAbbv": "DEN", "W": 54, "L": 28, "W_Pct": 0.659},
            {"teamAbbv": "LAL", "W": 53, "L": 29, "W_Pct": 0.646},
            {"teamAbbv": "HOU", "W": 52, "L": 30, "W_Pct": 0.634},
            {"teamAbbv": "MIN", "W": 49, "L": 33, "W_Pct": 0.598},
            {"teamAbbv": "PHX", "W": 45, "L": 37, "W_Pct": 0.549},
            {"teamAbbv": "POR", "W": 42, "L": 40, "W_Pct": 0.512},
            {"teamAbbv": "LAC", "W": 42, "L": 40, "W_Pct": 0.512},
            {"teamAbbv": "GSW", "W": 37, "L": 45, "W_Pct": 0.451},
            {"teamAbbv": "NOP", "W": 26, "L": 56, "W_Pct": 0.317},
            {"teamAbbv": "DAL", "W": 26, "L": 56, "W_Pct": 0.317},
            {"teamAbbv": "MEM", "W": 25, "L": 57, "W_Pct": 0.305},
            {"teamAbbv": "SAC", "W": 22, "L": 60, "W_Pct": 0.268},
            {"teamAbbv": "UTA", "W": 22, "L": 60, "W_Pct": 0.268},
        ],
    },
}


In [62]:
# Preseason vegas odds for each season
vegas_odds = {
    "21_22": {
        "BKN": {"odds": 240, "wl_ou": 56.5},
        "LAL": {"odds": 400, "wl_ou": 52.5},
        "MIL": {"odds": 900, "wl_ou": 54.5},
        "GSW": {"odds": 900, "wl_ou": 48.5},
        "PHX": {"odds": 1500, "wl_ou": 51.5},
        "UTA": {"odds": 1600, "wl_ou": 52.5},
        "PHI": {"odds": 1600, "wl_ou": 50.5},
        "LAC": {"odds": 1600, "wl_ou": 45.5},
        "DEN": {"odds": 2200, "wl_ou": 47.5},
        "MIA": {"odds": 2500, "wl_ou": 48.5},
        "DAL": {"odds": 3000, "wl_ou": 48.5},
        "ATL": {"odds": 3500, "wl_ou": 47.5},
        "BOS": {"odds": 4000, "wl_ou": 45.5},
        "POR": {"odds": 5000, "wl_ou": 44.5},
        "CHI": {"odds": 6600, "wl_ou": 42.5},
        "NYK": {"odds": 8000, "wl_ou": 41.5},
        "MEM": {"odds": 10000, "wl_ou": 41.5},
        "IND": {"odds": 10000, "wl_ou": 42.5},
        "NOP": {"odds": 10000, "wl_ou": 39.5},
        "CHA": {"odds": 10000, "wl_ou": 38.5},
        "WAS": {"odds": 15000, "wl_ou": 33.5},
        "TOR": {"odds": 15000, "wl_ou": 35.5},
        "SAC": {"odds": 25000, "wl_ou": 36.5},
        "MIN": {"odds": 25000, "wl_ou": 35.5},
        "SAS": {"odds": 30000, "wl_ou": 28.5},
        "CLE": {"odds": 50000, "wl_ou": 26.5},
        "ORL": {"odds": 50000, "wl_ou": 22.5},
        "OKC": {"odds": 50000, "wl_ou": 23.5},
        "DET": {"odds": 50000, "wl_ou": 24.5},
        "HOU": {"odds": 50000, "wl_ou": 24.5},
    },

    "22_23": {
        "BOS": {"odds": 500, "wl_ou": 54.5},
        "GSW": {"odds": 600, "wl_ou": 52.5},
        "MIL": {"odds": 650, "wl_ou": 53.5},
        "BKN": {"odds": 700, "wl_ou": 50.5},
        "LAC": {"odds": 700, "wl_ou": 52.5},
        "PHX": {"odds": 1000, "wl_ou": 52.5},
        "PHI": {"odds": 1400, "wl_ou": 50.5},
        "MIA": {"odds": 1600, "wl_ou": 49.5},
        "DEN": {"odds": 1800, "wl_ou": 51.5},
        "LAL": {"odds": 1800, "wl_ou": 44.5},
        "MEM": {"odds": 2000, "wl_ou": 49.5},
        "DAL": {"odds": 2500, "wl_ou": 48.5},
        "MIN": {"odds": 2800, "wl_ou": 49.5},
        "CLE": {"odds": 3000, "wl_ou": 46.5},
        "TOR": {"odds": 4000, "wl_ou": 46.5},
        "CHI": {"odds": 4000, "wl_ou": 41.5},
        "NOP": {"odds": 4000, "wl_ou": 45.5},
        "ATL": {"odds": 5000, "wl_ou": 46.5},
        "NYK": {"odds": 6600, "wl_ou": 38.5},
        "POR": {"odds": 8000, "wl_ou": 39.5},
        "CHA": {"odds": 15000, "wl_ou": 34.5},
        "WAS": {"odds": 25000, "wl_ou": 35.5},
        "DET": {"odds": 50000, "wl_ou": 29.5},
        "OKC": {"odds": 50000, "wl_ou": 23.5},
        "ORL": {"odds": 50000, "wl_ou": 26.5},
        "IND": {"odds": 50000, "wl_ou": 24.5},
        "SAC": {"odds": 50000, "wl_ou": 34.5},
        "SAS": {"odds": 50000, "wl_ou": 22.5},
        "HOU": {"odds": 50000, "wl_ou": 23.5},
        "UTA": {"odds": 50000, "wl_ou": 23.5},
    },

    "23_24": {
        "BOS": {"odds": 450, "wl_ou": 54.5},
        "DEN": {"odds": 450, "wl_ou": 52.5},
        "PHX": {"odds": 600, "wl_ou": 51.5},
        "MIL": {"odds": 600, "wl_ou": 54.5},
        "GSW": {"odds": 1200, "wl_ou": 48.5},
        "LAL": {"odds": 1500, "wl_ou": 47.5},
        "MIA": {"odds": 1600, "wl_ou": 45.5},
        "PHI": {"odds": 1800, "wl_ou": 48.5},
        "LAC": {"odds": 2200, "wl_ou": 46.5},
        "MEM": {"odds": 2200, "wl_ou": 45.5},
        "DAL": {"odds": 2500, "wl_ou": 45.5},
        "CLE": {"odds": 2500, "wl_ou": 50.5},
        "SAC": {"odds": 3500, "wl_ou": 44.5},
        "NOP": {"odds": 4000, "wl_ou": 44.5},
        "NYK": {"odds": 5000, "wl_ou": 45.5},
        "MIN": {"odds": 6600, "wl_ou": 44.5},
        "ATL": {"odds": 8000, "wl_ou": 42.5},
        "TOR": {"odds": 8000, "wl_ou": 36.5},
        "OKC": {"odds": 10000, "wl_ou": 44.5},
        "CHI": {"odds": 12500, "wl_ou": 37.5},
        "BKN": {"odds": 12500, "wl_ou": 37.5},
        "POR": {"odds": 12500, "wl_ou": 28.5},
        "SAS": {"odds": 15000, "wl_ou": 28.5},
        "UTA": {"odds": 25000, "wl_ou": 35.5},
        "ORL": {"odds": 25000, "wl_ou": 37.5},
        "IND": {"odds": 25000, "wl_ou": 38.5},
        "HOU": {"odds": 25000, "wl_ou": 31.5},
        "WAS": {"odds": 50000, "wl_ou": 24.5},
        "CHA": {"odds": 50000, "wl_ou": 31.5},
        "DET": {"odds": 50000, "wl_ou": 28.5},
    },

    "24_25": {
        "BOS": {"odds": 325, "wl_ou": 58.5},
        "OKC": {"odds": 675, "wl_ou": 57.5},
        "NYK": {"odds": 700, "wl_ou": 53.5},
        "PHI": {"odds": 1000, "wl_ou": 50.5},
        "DAL": {"odds": 1100, "wl_ou": 49.5},
        "DEN": {"odds": 1100, "wl_ou": 51.5},
        "MIN": {"odds": 1100, "wl_ou": 51.5},
        "MIL": {"odds": 1400, "wl_ou": 49.5},
        "PHX": {"odds": 1500, "wl_ou": 48.5},
        "CLE": {"odds": 3500, "wl_ou": 48.5},
        "MEM": {"odds": 3500, "wl_ou": 47.5},
        "ORL": {"odds": 4000, "wl_ou": 47.5},
        "GSW": {"odds": 4000, "wl_ou": 43.5},
        "LAL": {"odds": 4000, "wl_ou": 43.5},
        "MIA": {"odds": 5000, "wl_ou": 44.5},
        "NOP": {"odds": 5000, "wl_ou": 46.5},
        "IND": {"odds": 6600, "wl_ou": 46.5},
        "SAC": {"odds": 6600, "wl_ou": 46.5},
        "HOU": {"odds": 8000, "wl_ou": 43.5},
        "LAC": {"odds": 12500, "wl_ou": 35.5},
        "SAS": {"odds": 15000, "wl_ou": 35.5},
        "ATL": {"odds": 50000, "wl_ou": 36.5},
        "UTA": {"odds": 50000, "wl_ou": 28.5},
        "DET": {"odds": 100000, "wl_ou": 25.5},
        "CHA": {"odds": 100000, "wl_ou": 30.5},
        "CHI": {"odds": 100000, "wl_ou": 28.5},
        "BKN": {"odds": 100000, "wl_ou": 19.5},
        "POR": {"odds": 100000, "wl_ou": 21.5},
        "TOR": {"odds": 100000, "wl_ou": 29.5},
        "WAS": {"odds": 100000, "wl_ou": 20.5},
    },

    "25_26": {
        "OKC": {"odds": 240, "wl_ou": 62.5},
        "DEN": {"odds": 550, "wl_ou": 53.5},
        "CLE": {"odds": 750, "wl_ou": 56.5},
        "NYK": {"odds": 900, "wl_ou": 53.5},
        "MIN": {"odds": 1300, "wl_ou": 49.5},
        "HOU": {"odds": 1400, "wl_ou": 52.5},
        "LAL": {"odds": 1600, "wl_ou": 46.5},
        "LAC": {"odds": 1800, "wl_ou": 49.5},
        "ORL": {"odds": 1800, "wl_ou": 51.5},
        "GSW": {"odds": 2500, "wl_ou": 47.5},
        "DET": {"odds": 3300, "wl_ou": 46.5},
        "DAL": {"odds": 3500, "wl_ou": 41.5},
        "PHI": {"odds": 4000, "wl_ou": 43.5},
        "ATL": {"odds": 4000, "wl_ou": 46.5},
        "MIL": {"odds": 5500, "wl_ou": 43.5},
        "BOS": {"odds": 6000, "wl_ou": 41.5},
        "SAS": {"odds": 6600, "wl_ou": 44.5},
        "IND": {"odds": 10000, "wl_ou": 37.5},
        "TOR": {"odds": 10000, "wl_ou": 39.5},
        "MEM": {"odds": 12500, "wl_ou": 39.5},
        "MIA": {"odds": 20000, "wl_ou": 37.5},
        "NOP": {"odds": 30000, "wl_ou": 30.5},
        "PHX": {"odds": 50000, "wl_ou": 30.5},
        "POR": {"odds": 50000, "wl_ou": 35.5},
        "SAC": {"odds": 50000, "wl_ou": 35.5},
        "CHI": {"odds": 50000, "wl_ou": 33.5},
        "CHA": {"odds": 100000, "wl_ou": 27.5},
        "BKN": {"odds": 100000, "wl_ou": 19.5},
        "UTA": {"odds": 100000, "wl_ou": 18.5},
        "WAS": {"odds": 100000, "wl_ou": 20.5},
    },

    "26_27": {
        "ATL": {"odds": -105, "wl_ou": 43.5},
        "BOS": {"odds": -115, "wl_ou": 51.5},
        "BKN": {"odds": 100, "wl_ou": 24.5},
        "CHA": {"odds": -120, "wl_ou": 37.5},
        "CHI": {"odds": -135, "wl_ou": 27.5},
        "CLE": {"odds": -115, "wl_ou": 47.5},
        "DAL": {"odds": -110, "wl_ou": 34.5},
        "DEN": {"odds": 105, "wl_ou": 49.5},
        "DET": {"odds": -105, "wl_ou": 49.5},
        "GSW": {"odds": 100, "wl_ou": 40.5},
        "HOU": {"odds": -110, "wl_ou": 47.5},
        "IND": {"odds": -125, "wl_ou": 44.5},
        "LAC": {"odds": 100, "wl_ou": 30.5},
        "LAL": {"odds": -105, "wl_ou": 46.5},
        "MEM": {"odds": -120, "wl_ou": 28.5},
        "MIA": {"odds": 105, "wl_ou": 46.5},
        "MIL": {"odds": -115, "wl_ou": 25.5},
        "MIN": {"odds": -110, "wl_ou": 48.5},
        "NOP": {"odds": -115, "wl_ou": 27.5},
        "NYK": {"odds": 100, "wl_ou": 52.5},
        "OKC": {"odds": -105, "wl_ou": 62.5},
        "ORL": {"odds": -130, "wl_ou": 43.5},
        "PHI": {"odds": 100, "wl_ou": 50.5},
        "PHX": {"odds": -120, "wl_ou": 38.5},
        "POR": {"odds": 100, "wl_ou": 42.5},
        "SAS": {"odds": -115, "wl_ou": 59.5},
        "SAC": {"odds": -105, "wl_ou": 21.5},
        "TOR": {"odds": -125, "wl_ou": 45.5},
        "UTA": {"odds": -145, "wl_ou": 35.5},
        "WAS": {"odds": -105, "wl_ou": 34.5},
    },
}

In [63]:
# Create dataframe to compare standings

season = "26_27"

# WEST


# Model standings
model_west = (
    pd.DataFrame(results[season]["standings"]["West"])
    .T
    .reset_index()
    .rename(columns={
        "index": "Team",
        "W_Pct": "W%"
    })
)

# Keep only the relevant columns
model_west = model_west[["Team", "W"]].rename(
    columns={"W": "Model Wins"}
)

# Adjust model wins:
# Winning record -> 2 additional losses
# Losing record -> 2 additional wins
model_west.loc[model_west["Model Wins"] > 41, "Model Wins"] -= 2
model_west.loc[model_west["Model Wins"] < 41, "Model Wins"] += 2



# Actual standings
actual_west = (
    pd.DataFrame(nba_standings[season]["west"])
    .rename(columns={
        "teamAbbv": "Team",
        "W_Pct": "W%"
    })
)

actual_west = actual_west[["Team", "W"]].rename(
    columns={"W": "Actual Wins"}
)

# Vegas
vegas_west = (
    pd.DataFrame.from_dict(
        vegas_odds[season],
        orient="index"
    )
    .reset_index()
    .rename(columns={
        "index": "Team",
        "wl_ou": "W-L O/U"
    })
)

vegas_west = vegas_west[["Team", "W-L O/U"]]

# Merge
west_final = (
    model_west
    .merge(vegas_west, on="Team", how="left")
    .merge(actual_west, on="Team", how="left")
)

# Differences
west_final["Model - Vegas"] = (
    west_final["Model Wins"] - west_final["W-L O/U"]
)

west_final["Actual - Vegas"] = (
    west_final["Actual Wins"] - west_final["W-L O/U"]
)


# Final Column Order
west_final = west_final[
    [
        "Team",
        "Model Wins",
        "W-L O/U",
        "Model - Vegas",
        "Actual - Vegas",
        "Actual Wins"
    ]
]

# Sort by actual standings
west_final = (
    west_final
    .sort_values("Model Wins", ascending=False)
    .reset_index(drop=True)
)
# Clean numeric formatting
west_final["Model Wins"] = west_final["Model Wins"].astype(int)
west_final["Actual Wins"] = west_final["Actual Wins"].astype(int)
west_final["W-L O/U"] = west_final["W-L O/U"].astype(float)

west_final["Model - Vegas"] = west_final["Model - Vegas"].round(1)
west_final["Actual - Vegas"] = west_final["Actual - Vegas"].round(1)

# EAST

# Model standings
model_east = (
    pd.DataFrame(results[season]["standings"]["East"])
    .T
    .reset_index()
    .rename(columns={
        "index": "Team",
        "W_Pct": "W%"
    })
)

model_east = model_east[["Team", "W"]].rename(
    columns={"W": "Model Wins"}
)

# Adjust model wins:
# Winning record -> 2 additional losses
# Losing record -> 2 additional wins
model_east.loc[model_east["Model Wins"] > 41, "Model Wins"] -= 2
model_east.loc[model_east["Model Wins"] < 41, "Model Wins"] += 2


# Actual standings
actual_east = (
    pd.DataFrame(nba_standings[season]["east"])
    .rename(columns={
        "teamAbbv": "Team",
        "W_Pct": "W%"
    })
)

actual_east = actual_east[["Team", "W"]].rename(
    columns={"W": "Actual Wins"}
)


# Vegas
vegas_east = (
    pd.DataFrame.from_dict(
        vegas_odds[season],
        orient="index"
    )
    .reset_index()
    .rename(columns={
        "index": "Team",
        "wl_ou": "W-L O/U"
    })
)

vegas_east = vegas_east[["Team", "W-L O/U"]]

# Merge
east_final = (
    model_east
    .merge(vegas_east, on="Team", how="left")
    .merge(actual_east, on="Team", how="left")
)

# Differences
east_final["Model - Vegas"] = (
    east_final["Model Wins"] - east_final["W-L O/U"]
)

east_final["Actual - Vegas"] = (
    east_final["Actual Wins"] - east_final["W-L O/U"]
)

# Final Column Order
east_final = east_final[
    [
        "Team",
        "Model Wins",
        "W-L O/U",
        "Model - Vegas",
        "Actual - Vegas",
        "Actual Wins"
    ]
]

# Sort by actual standings
east_final = (
    east_final
    .sort_values("Model Wins", ascending=False)
    .reset_index(drop=True)
)

# Formatting
east_final["Model Wins"] = east_final["Model Wins"].astype(int)
east_final["Actual Wins"] = east_final["Actual Wins"].astype(int)
east_final["W-L O/U"] = east_final["W-L O/U"].astype(float)

east_final["Model - Vegas"] = east_final["Model - Vegas"].round(1)
east_final["Actual - Vegas"] = east_final["Actual - Vegas"].round(1)

In [64]:
# Compare model vs vegas accuracies West
west_error = west_final.copy()

# Absolute error from actual win total
west_error["Model Abs Error"] = (
    west_error["Model Wins"] - west_error["Actual Wins"]
).abs()

west_error["Vegas Abs Error"] = (
    west_error["W-L O/U"] - west_error["Actual Wins"]
).abs()


# Summary statistics
west_stats = pd.DataFrame({
    "Mean": [
        west_error["Model Abs Error"].mean(),
        west_error["Vegas Abs Error"].mean()
    ],
    "Median": [
        west_error["Model Abs Error"].median(),
        west_error["Vegas Abs Error"].median()
    ],
    "Std Dev": [
        west_error["Model Abs Error"].std(),
        west_error["Vegas Abs Error"].std()
    ],
    "Min": [
        west_error["Model Abs Error"].min(),
        west_error["Vegas Abs Error"].min()
    ],
    "Max": [
        west_error["Model Abs Error"].max(),
        west_error["Vegas Abs Error"].max()
    ]
}, index=["Model", "Vegas"]).round(2)

display(west_final)
display(west_stats)

,Team,Model Wins,W-L O/U,Model - Vegas,Actual - Vegas,Actual Wins
0,SAS,63,59.5,3.5,2.5,62
1,OKC,59,62.5,-3.5,1.5,64
2,POR,57,42.5,14.5,-0.5,42
3,LAL,55,46.5,8.5,6.5,53
4,GSW,46,40.5,5.5,-3.5,37
5,DEN,41,49.5,-8.5,4.5,54
6,MIN,41,48.5,-7.5,0.5,49
7,MEM,34,28.5,5.5,-3.5,25
8,NOP,31,27.5,3.5,-1.5,26
9,PHX,30,38.5,-8.5,6.5,45


,Mean,Median,Std Dev,Min,Max
Model,10.33,9.0,8.49,1.0,31.0
Vegas,4.63,3.5,4.02,0.5,13.5


In [65]:
# Compare model vs vegas accuracies East

east_error = east_final.copy()

# Absolute error from actual win total
east_error["Model Abs Error"] = (
    east_error["Model Wins"] - east_error["Actual Wins"]
).abs()

east_error["Vegas Abs Error"] = (
    east_error["W-L O/U"] - east_error["Actual Wins"]
).abs()


# Summary statistics
east_stats = pd.DataFrame({
    "Mean": [
        east_error["Model Abs Error"].mean(),
        east_error["Vegas Abs Error"].mean()
    ],
    "Median": [
        east_error["Model Abs Error"].median(),
        east_error["Vegas Abs Error"].median()
    ],
    "Std Dev": [
        east_error["Model Abs Error"].std(),
        east_error["Vegas Abs Error"].std()
    ],
    "Min": [
        east_error["Model Abs Error"].min(),
        east_error["Vegas Abs Error"].min()
    ],
    "Max": [
        east_error["Model Abs Error"].max(),
        east_error["Vegas Abs Error"].max()
    ]
}, index=["Model", "Vegas"]).round(2)

display(east_final)
display(east_stats)

,Team,Model Wins,W-L O/U,Model - Vegas,Actual - Vegas,Actual Wins
0,NYK,61,52.5,8.5,0.5,53
1,MIA,57,46.5,10.5,-3.5,43
2,PHI,57,50.5,6.5,-5.5,45
3,BOS,55,51.5,3.5,4.5,56
4,CLE,49,47.5,1.5,4.5,52
5,ORL,46,43.5,2.5,1.5,45
6,IND,45,44.5,0.5,-25.5,19
7,ATL,42,43.5,-1.5,2.5,46
8,WAS,42,34.5,7.5,-17.5,17
9,CHA,41,37.5,3.5,6.5,44


,Mean,Median,Std Dev,Min,Max
Model,9.53,5.0,8.70,1.0,26.0
Vegas,6.50,4.5,6.77,0.5,25.5
